# BIST-100 Güniçi Scalp Trading: TA + GARCH + XGBoost Şampiyon Strateji Motoru

**Mimari:** Özellik Mühendisliği → Üç Bağımsız Strateji Motoru → Vektörel Backtest → Kıyaslama & Şampiyon İlanı

| Katman | Açıklama |
|--------|----------|
| Veri | yfinance ile 1d/1h/15m OHLCV + XU100 endeks verisi |
| Strateji A | RSI-14 + MACD sinyal kesişimi (kural tabanlı TA) |
| Strateji B | GARCH(1,1) koşullu oynaklık eşik filtresi |
| Strateji C | XGBoost sınıflandırıcı (teknik + göreceli özellikler) |
| Backtest | Kronolojik %20 test seti, vektörel hesaplama |
| Metrikler | Getiri, Sharpe, MaxDD, Win Rate, Calmar |

In [ ]:
# Gerekli kütüphaneleri kur
import subprocess, sys

pkgs = [
    'git+https://github.com/rongardF/tvdatafeed.git',
    'arch', 'xgboost', 'ta', 'matplotlib', 'seaborn',
    'scikit-learn', 'optuna', 'joblib',
]
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', p, '-q'], check=False)

print('Kurulum tamamlandı.')


In [ ]:
import warnings, json, os, hashlib
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path

# TradingView veri kaynağı
from tvDatafeed import TvDatafeed, Interval

# İstatistik & ML
from arch import arch_model
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Teknik analiz
import ta

pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('dark_background')
COLORS = ['#00ff88', '#ff6b6b', '#4ecdc4', '#ffd93d', '#c77dff']

# Hafıza dosyaları (modeli ve geçmiş performansı saklar)
MEMORY_DIR   = Path('bist_memory')
MEMORY_DIR.mkdir(exist_ok=True)
MODEL_FILE   = MEMORY_DIR / 'xgb_model.pkl'
SCALER_FILE  = MEMORY_DIR / 'scaler.pkl'
HISTORY_FILE = MEMORY_DIR / 'performance_history.json'
PARAMS_FILE  = MEMORY_DIR / 'best_params.json'

print('Tüm kütüphaneler yüklendi ✓')
print(f'Hafıza klasörü: {MEMORY_DIR.resolve()}')


## ⚙️ KONFIGÜRASYON — Hisseyi, Zaman Dilimini ve Parametreleri Buradan Ayarla

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  KULLANICI AYARLARI  — sadece bu bloğu değiştir
# ═══════════════════════════════════════════════════════════════

# TradingView sembol ve borsa adı (tvdatafeed formatı)
TICKER        = 'THYAO'          # TradingView sembol adı (büyük harf, .IS eki olmadan)
INDEX_TKR     = 'XU100'          # Endeks sembol adı
EXCHANGE      = 'BIST'           # Borsa adı (TradingView'deki gibi)
INDEX_EXCHANGE= 'BIST'

# TradingView oturum açma (opsiyonel — boş bırakılırsa misafir erişim kullanılır)
TV_USERNAME   = ''               # TradingView kullanıcı adı (boş bırakılabilir)
TV_PASSWORD   = ''               # TradingView şifresi     (boş bırakılabilir)

# Zaman dilimi → Interval enum eşlemesi
# Seçenekler: '1m','3m','5m','15m','30m','45m','1h','2h','3h','4h','1d','1W','1M'
INTERVAL_STR  = '1h'

TV_INTERVAL_MAP = {
    '1m' : Interval.in_1_minute,
    '3m' : Interval.in_3_minute,
    '5m' : Interval.in_5_minute,
    '15m': Interval.in_15_minute,
    '30m': Interval.in_30_minute,
    '45m': Interval.in_45_minute,
    '1h' : Interval.in_1_hour,
    '2h' : Interval.in_2_hour,
    '3h' : Interval.in_3_hour,
    '4h' : Interval.in_4_hour,
    '1d' : Interval.in_daily,
    '1W' : Interval.in_weekly,
    '1M' : Interval.in_monthly,
}
TV_INTERVAL = TV_INTERVAL_MAP[INTERVAL_STR]
INTERVAL    = INTERVAL_STR   # backtest annualize faktörü için

# Kaç bar çekilsin
N_BARS        = 5000           # tvdatafeed maksimum bar sayısı

# Teknik parametreler
RSI_PERIOD   = 14
MACD_FAST    = 12
MACD_SLOW    = 26
MACD_SIGNAL  = 9
VOL_MA_WIN   = 20

# GARCH oynaklık eşiği (yüzdelik dilim)
GARCH_THRESH_LOW  = 30
GARCH_THRESH_HIGH = 70

# XGBoost
TRAIN_RATIO  = 0.80
XGB_PARAMS   = dict(
    n_estimators=400, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1
)

# Backtest
TRANSACTION_COST = 0.001
INITIAL_CAPITAL  = 100_000

print(f'Konfigürasyon: {TICKER} ({EXCHANGE}) | {INTERVAL_STR} | {N_BARS} bar')


## 📥 1. VERİ İNDİRME

In [ ]:
def get_tv_client() -> TvDatafeed:
    """TvDatafeed istemcisini oluştur. Kullanıcı adı/şifre boşsa misafir modda çalışır."""
    if TV_USERNAME and TV_PASSWORD:
        tv = TvDatafeed(TV_USERNAME, TV_PASSWORD)
        print('TradingView oturumu açıldı ✓')
    else:
        tv = TvDatafeed()
        print('TradingView misafir modda bağlandı ✓ (giriş yapılmadı)')
    return tv


def tv_to_df(raw: pd.DataFrame) -> pd.DataFrame:
    """
    tvdatafeed çıktısını standart OHLCV DataFrame'e dönüştür.
    tvdatafeed sütun adları: symbol, open, high, low, close, volume
    """
    df = raw.copy()
    df.index = pd.to_datetime(df.index)
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)

    # Sütun adlarını büyük harfe çevir
    df.columns = [c.capitalize() for c in df.columns]

    # 'Symbol' sütunu varsa düşür (tvdatafeed ekler)
    if 'Symbol' in df.columns:
        df.drop(columns=['Symbol'], inplace=True)

    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    df.sort_index(inplace=True)
    df.dropna(subset=['Close'], inplace=True)
    return df


def download_data(ticker: str, exchange: str,
                  index_ticker: str, index_exchange: str,
                  interval: Interval, n_bars: int) -> pd.DataFrame:
    """
    tvdatafeed ile hisse + endeks verisini çek, birleştir ve temizle.
    """
    tv = get_tv_client()

    print(f'▶ {ticker} ({exchange}) indiriliyor...')
    raw_stock = tv.get_hist(
        symbol=ticker, exchange=exchange,
        interval=interval, n_bars=n_bars
    )
    if raw_stock is None or raw_stock.empty:
        raise ValueError(f'{ticker} için veri alınamadı. Sembol/borsa adını kontrol et.')

    print(f'▶ {index_ticker} ({index_exchange}) indiriliyor...')
    raw_index = tv.get_hist(
        symbol=index_ticker, exchange=index_exchange,
        interval=interval, n_bars=n_bars
    )

    df = tv_to_df(raw_stock)

    if raw_index is not None and not raw_index.empty:
        idx = tv_to_df(raw_index)
        df['Endeks_Close'] = idx['Close'].reindex(df.index, method='ffill')
    else:
        print('Uyarı: Endeks verisi alınamadı, hisse fiyatı kullanılıyor.')
        df['Endeks_Close'] = df['Close']

    print(f'✓ {len(df):,} bar yüklendi | {df.index[0].date()} → {df.index[-1].date()}')
    return df


# Veriyi çek
df_raw = download_data(
    TICKER, EXCHANGE,
    INDEX_TKR, INDEX_EXCHANGE,
    TV_INTERVAL, N_BARS
)
df_raw.tail()


## 🔧 2. ÖZELLİK MÜHENDİSLİĞİ & HEDEF TANIMLAMA

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Tüm teknik ve istatistiksel özellikleri hesapla."""
    d = df.copy()

    # ── Temel getiri ──────────────────────────────────────────────
    d['Log_Return']  = np.log(d['Close'] / d['Close'].shift(1))
    d['Next_Return'] = d['Log_Return'].shift(-1)   # hedef için
    d['Label']       = (d['Next_Return'] > 0).astype(int)

    # ── RSI ───────────────────────────────────────────────────────
    d['RSI'] = ta.momentum.RSIIndicator(close=d['Close'], window=RSI_PERIOD).rsi()

    # ── MACD ──────────────────────────────────────────────────────
    macd_obj = ta.trend.MACD(
        close=d['Close'],
        window_slow=MACD_SLOW,
        window_fast=MACD_FAST,
        window_sign=MACD_SIGNAL
    )
    d['MACD']        = macd_obj.macd()
    d['MACD_Signal'] = macd_obj.macd_signal()
    d['MACD_Hist']   = macd_obj.macd_diff()

    # ── Hareketli Ortalamalar ─────────────────────────────────────
    d['EMA9']  = ta.trend.EMAIndicator(d['Close'], window=9).ema_indicator()
    d['EMA21'] = ta.trend.EMAIndicator(d['Close'], window=21).ema_indicator()
    d['SMA50'] = ta.trend.SMAIndicator(d['Close'], window=50).sma_indicator()
    d['SMA200']= ta.trend.SMAIndicator(d['Close'], window=200).sma_indicator()

    # ── Bollinger Bantları ────────────────────────────────────────
    bb = ta.volatility.BollingerBands(d['Close'], window=20, window_dev=2)
    d['BB_High'] = bb.bollinger_hband()
    d['BB_Low']  = bb.bollinger_lband()
    d['BB_Width']= bb.bollinger_wband()
    d['BB_Pct']  = bb.bollinger_pband()   # 0–1 arası pozisyon

    # ── ATR (Average True Range) ──────────────────────────────────
    d['ATR'] = ta.volatility.AverageTrueRange(
        d['High'], d['Low'], d['Close'], window=14
    ).average_true_range()
    d['ATR_Pct'] = d['ATR'] / d['Close']  # normalize

    # ── Stokastik ─────────────────────────────────────────────────
    stoch = ta.momentum.StochasticOscillator(d['High'], d['Low'], d['Close'], window=14, smooth_window=3)
    d['Stoch_K'] = stoch.stoch()
    d['Stoch_D'] = stoch.stoch_signal()

    # ── Hacim Özellikleri ─────────────────────────────────────────
    d['Vol_MA']     = d['Volume'].rolling(VOL_MA_WIN).mean()
    d['Vol_Ratio']  = d['Volume'] / d['Vol_MA']        # hacim baskısı
    d['OBV']        = ta.volume.OnBalanceVolumeIndicator(d['Close'], d['Volume']).on_balance_volume()
    d['OBV_EMA']    = d['OBV'].ewm(span=20).mean()
    d['OBV_Signal'] = (d['OBV'] > d['OBV_EMA']).astype(int)

    # ── Endeks Göreceli Gücü ──────────────────────────────────────
    d['Relative_Strength'] = d['Close'] / d['Endeks_Close']
    d['RS_Momentum']       = d['Relative_Strength'].pct_change(5)  # 5 bar RS değişimi

    # ── Momentum / ROC ────────────────────────────────────────────
    d['ROC5']  = ta.momentum.ROCIndicator(d['Close'], window=5).roc()
    d['ROC10'] = ta.momentum.ROCIndicator(d['Close'], window=10).roc()
    d['ROC20'] = ta.momentum.ROCIndicator(d['Close'], window=20).roc()

    # ── Trend Gücü ────────────────────────────────────────────────
    adx = ta.trend.ADXIndicator(d['High'], d['Low'], d['Close'], window=14)
    d['ADX']   = adx.adx()
    d['DI_pos']= adx.adx_pos()
    d['DI_neg']= adx.adx_neg()

    # ── Fiyat Pattern Özellikleri ─────────────────────────────────
    d['Body_Size']   = abs(d['Close'] - d['Open']) / (d['High'] - d['Low'] + 1e-9)
    d['Upper_Wick']  = (d['High'] - d[['Close','Open']].max(axis=1)) / (d['High'] - d['Low'] + 1e-9)
    d['Lower_Wick']  = (d[['Close','Open']].min(axis=1) - d['Low']) / (d['High'] - d['Low'] + 1e-9)
    d['Bullish_Bar'] = (d['Close'] > d['Open']).astype(int)

    # ── Gecikme (Lag) Özellikleri ─────────────────────────────────
    for lag in [1, 2, 3, 5]:
        d[f'Ret_lag{lag}'] = d['Log_Return'].shift(lag)
        d[f'RSI_lag{lag}'] = d['RSI'].shift(lag)

    # ── Güniçi Volatilite ─────────────────────────────────────────
    d['Intraday_Range'] = (d['High'] - d['Low']) / d['Open']
    d['Gap_Pct']        = (d['Open'] - d['Close'].shift(1)) / d['Close'].shift(1)

    return d


df = build_features(df_raw)
print(f'Özellik sayısı: {df.shape[1]} | Satır: {df.shape[0]:,}')
df[['Close','RSI','MACD','MACD_Signal','ATR_Pct','Vol_Ratio','Label']].tail(10)

## 🏗️ 3. STRATEJİ MOTORLARI

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  STRATEJİ A — Klasik Teknik Analiz (RSI + MACD + EMA Filtresi)
# ══════════════════════════════════════════════════════════════════

def strategy_A_signals(df: pd.DataFrame) -> pd.Series:
    """
    Al sinyali  (1) : RSI < 70 VE RSI > 30 VE MACD histogram pozitife döndü
                       VE fiyat EMA9 > EMA21 (kısa vadeli trend yukarı)
    Sat sinyali (-1): RSI > 70 VEYA MACD histogram negatife döndü
                       VEYA fiyat EMA9 < EMA21
    """
    hist         = df['MACD_Hist']
    hist_cross_up = (hist > 0) & (hist.shift(1) <= 0)   # histogram sıfırı yukarı kesiyor
    hist_cross_dn = (hist < 0) & (hist.shift(1) >= 0)

    rsi_ok    = (df['RSI'] > 30) & (df['RSI'] < 70)
    trend_up  = df['EMA9'] > df['EMA21']
    trend_dn  = df['EMA9'] < df['EMA21']

    # Güçlendirici filtreler
    vol_confirm   = df['Vol_Ratio'] > 1.0      # hacim ortalamanın üzerinde
    adx_trending  = df['ADX'] > 20             # yeterli trend gücü

    sig = pd.Series(0, index=df.index)
    sig[hist_cross_up & rsi_ok & trend_up & vol_confirm] = 1
    sig[hist_cross_dn | trend_dn]                        = -1

    return sig


print('Strateji A (Klasik TA) fonksiyonu tanımlandı ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  STRATEJİ B — GARCH(1,1) Oynaklık Filtresi
# ══════════════════════════════════════════════════════════════════

def strategy_B_signals(df: pd.DataFrame,
                        thresh_low: float = GARCH_THRESH_LOW,
                        thresh_high: float = GARCH_THRESH_HIGH) -> pd.Series:
    """
    GARCH(1,1) ile koşullu oynaklık (σ_t) hesapla.
    - Oynaklık percentile DÜŞÜK bölgedeyse (sıkışma / fırlatma öncesi):
        RSI yönüne göre al (>50) veya sat (<50)
    - Oynaklık percentile YÜKSEK bölgedeyse:
        Piyasadan çık (0) — yüksek riskten kaçın
    """
    returns = df['Log_Return'].dropna() * 100  # % cinsinden

    # GARCH modelini fit et
    try:
        model  = arch_model(returns, vol='Garch', p=1, q=1, dist='Normal', rescale=False)
        result = model.fit(disp='off', options={'maxiter': 500})
        cond_vol = result.conditional_volatility  # % cinsinden
    except Exception as e:
        print(f'GARCH fit hatası: {e}. Sabit oynaklık kullanılıyor.')
        cond_vol = pd.Series(returns.rolling(20).std().values, index=returns.index)

    # Yüzdelik dilim hesapla (rolling 252 bar)
    vol_pct = cond_vol.rolling(252, min_periods=50).rank(pct=True) * 100
    vol_pct = vol_pct.reindex(df.index)  # df ile hizala

    low_vol  = vol_pct < thresh_low
    high_vol = vol_pct > thresh_high

    # Yön için RSI ve MACD histogram kullan
    bullish = (df['RSI'] > 50) & (df['MACD_Hist'] > 0) & (df['EMA9'] > df['EMA21'])
    bearish = (df['RSI'] < 50) & (df['MACD_Hist'] < 0)

    sig = pd.Series(0, index=df.index)
    sig[low_vol & bullish]  =  1   # sıkışma → yukarı kırılım beklentisi
    sig[low_vol & bearish]  = -1   # sıkışma → aşağı kırılım beklentisi
    sig[high_vol]           =  0   # yüksek oynaklık → bekle

    # GARCH oynaklık serisini de döndür (görselleştirme için)
    df['GARCH_Vol']     = cond_vol.reindex(df.index)
    df['GARCH_Vol_Pct'] = vol_pct

    return sig


print('Strateji B (GARCH) fonksiyonu tanımlandı ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  STRATEJİ C — XGBoost Sınıflandırıcı
# ══════════════════════════════════════════════════════════════════

FEATURE_COLS = [
    'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist',
    'EMA9', 'EMA21', 'BB_Pct', 'BB_Width',
    'ATR_Pct', 'Stoch_K', 'Stoch_D',
    'Vol_Ratio', 'OBV_Signal',
    'Relative_Strength', 'RS_Momentum',
    'ROC5', 'ROC10', 'ROC20',
    'ADX', 'DI_pos', 'DI_neg',
    'Body_Size', 'Upper_Wick', 'Lower_Wick', 'Bullish_Bar',
    'Ret_lag1', 'Ret_lag2', 'Ret_lag3', 'Ret_lag5',
    'RSI_lag1', 'RSI_lag2', 'RSI_lag3',
    'Intraday_Range', 'Gap_Pct',
]

def strategy_C_train_predict(df: pd.DataFrame, train_ratio: float = TRAIN_RATIO):
    """
    Kronolojik bölme: train_ratio kadarı eğitim, kalanı test.
    Döndürür: (sinyaller, test_start_idx, model, scaler, feature_importance)
    """
    avail_cols = [c for c in FEATURE_COLS if c in df.columns]
    dfc = df[avail_cols + ['Label']].dropna().copy()

    split_n    = int(len(dfc) * train_ratio)
    train_data = dfc.iloc[:split_n]
    test_data  = dfc.iloc[split_n:]

    X_train, y_train = train_data[avail_cols], train_data['Label']
    X_test,  y_test  = test_data[avail_cols],  test_data['Label']

    scaler  = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    # Sınıf dengesi için ağırlık
    pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model = XGBClassifier(**XGB_PARAMS, scale_pos_weight=pos_weight)
    model.fit(
        X_train_sc, y_train,
        eval_set=[(X_test_sc, y_test)],
        verbose=False
    )

    preds_train = model.predict(X_train_sc)
    preds_test  = model.predict(X_test_sc)
    proba_test  = model.predict_proba(X_test_sc)[:, 1]

    train_acc = accuracy_score(y_train, preds_train)
    test_acc  = accuracy_score(y_test,  preds_test)
    print(f'XGBoost | Train Acc: {train_acc:.3f} | Test Acc: {test_acc:.3f}')
    print(f'Test dönemi: {test_data.index[0].date()} → {test_data.index[-1].date()} ({len(test_data):,} bar)')

    # Sinyal serisi: test döneminde tahminleri al, geri kalanı 0
    sig = pd.Series(0, index=df.index)
    # 1 = al tahmini, 0 = sat/bekle tahmini → -1 olarak kodla
    sig.loc[test_data.index] = np.where(preds_test == 1, 1, -1)

    feat_imp = pd.Series(model.feature_importances_, index=avail_cols).sort_values(ascending=False)

    return sig, test_data.index[0], model, scaler, feat_imp, test_acc


print('Strateji C (XGBoost) fonksiyonu tanımlandı ✓')

## 🚀 4. STRATEJİLERİ ÇALIŞTIR

In [ ]:
print('═'*60)
print('STRATEJİLER HESAPLANIYOR...')
print('═'*60)

# Strateji A
print('\n[A] Klasik TA sinyalleri hesaplanıyor...')
sig_A = strategy_A_signals(df)
print(f'    Sinyal dağılımı: Al={( sig_A==1).sum()} | Sat={(sig_A==-1).sum()} | Bekle={(sig_A==0).sum()}')

# Strateji B
print('\n[B] GARCH(1,1) fit ediliyor...')
sig_B = strategy_B_signals(df)
print(f'    Sinyal dağılımı: Al={( sig_B==1).sum()} | Sat={(sig_B==-1).sum()} | Bekle={(sig_B==0).sum()}')

# Strateji C
print('\n[C] XGBoost eğitiliyor...')
sig_C, test_start, xgb_model, xgb_scaler, feat_imp, xgb_test_acc = strategy_C_train_predict(df)

print(f'\n✓ Tüm stratejiler hazır. Test dönemi başlangıcı: {test_start}')

## 📊 5. BACKTEST & KIYASLAMA MOTORU

In [ ]:
def run_backtest(df: pd.DataFrame, signals: pd.Series, strategy_name: str,
                 start_date=None, cost: float = TRANSACTION_COST) -> dict:
    """
    Vektörel backtest motoru.
    - Sinyal 1  → uzun pozisyon (bir sonraki barda giriş)
    - Sinyal -1 → kısa pozisyon (BIST'te CFD/Vadeli ile uygulanabilir; spot için 0 al)
    - Sinyal 0  → nakit
    """
    data = df.copy()
    data['Signal'] = signals

    # Test dönemine kırp
    if start_date is not None:
        data = data[data.index >= start_date]

    data = data.dropna(subset=['Log_Return'])

    # Bir sonraki barda işlem yap (look-ahead bias önlemi)
    data['Position'] = data['Signal'].shift(1).fillna(0).clip(-1, 1)

    # Ham getiri (pozisyon × bar getirisi)
    data['Strat_Return'] = data['Position'] * data['Log_Return']

    # İşlem maliyeti: pozisyon değiştiğinde uygula
    pos_change = data['Position'].diff().abs()
    data['Cost'] = pos_change * cost
    data['Net_Return'] = data['Strat_Return'] - data['Cost']

    # Kümülatif getiri
    data['Cum_Return']  = np.exp(data['Net_Return'].cumsum())
    data['BH_Return']   = np.exp(data['Log_Return'].cumsum())   # Buy & Hold kıyası

    # ── Metrikler ─────────────────────────────────────────────────
    total_ret = data['Cum_Return'].iloc[-1] - 1

    # Annualize faktörü (bar başına)
    bars_per_year = {'1d': 252, '1h': 252*7, '15m': 252*28, '5m': 252*78}.get(INTERVAL, 252)
    mean_ret = data['Net_Return'].mean()
    std_ret  = data['Net_Return'].std() + 1e-9
    sharpe   = (mean_ret / std_ret) * np.sqrt(bars_per_year)

    # Max Drawdown
    roll_max = data['Cum_Return'].cummax()
    drawdown = data['Cum_Return'] / roll_max - 1
    max_dd   = drawdown.min()

    # Calmar
    calmar = (total_ret / abs(max_dd)) if max_dd != 0 else np.nan

    # Win Rate: pozisyon açık barlarda kazanma yüzdesi
    active   = data[data['Position'] != 0]
    win_rate = (active['Net_Return'] > 0).mean() * 100 if len(active) > 0 else np.nan

    # İşlem sayısı
    trades = int(pos_change.sum() / 2)

    # Buy & Hold getirisi (kıyas)
    bh_ret = data['BH_Return'].iloc[-1] - 1

    return {
        'Strateji'       : strategy_name,
        'Toplam Getiri %': round(total_ret * 100, 2),
        'BH Getiri %'    : round(bh_ret * 100, 2),
        'Sharpe'         : round(sharpe, 3),
        'Max Drawdown %' : round(max_dd * 100, 2),
        'Calmar'         : round(calmar, 3) if not np.isnan(calmar) else 'N/A',
        'Win Rate %'     : round(win_rate, 2),
        'İşlem Sayısı'   : trades,
        '_data'          : data,   # görselleştirme için
    }


print('Backtest motoru tanımlandı ✓')

In [ ]:
# Tüm stratejileri test döneminde backtest et
res_A = run_backtest(df, sig_A, 'A: Klasik TA',   start_date=test_start)
res_B = run_backtest(df, sig_B, 'B: GARCH',       start_date=test_start)
res_C = run_backtest(df, sig_C, 'C: XGBoost',     start_date=test_start)

results = [res_A, res_B, res_C]

# Kıyaslama tablosu
metric_cols = ['Strateji','Toplam Getiri %','BH Getiri %','Sharpe',
               'Max Drawdown %','Calmar','Win Rate %','İşlem Sayısı']
comparison_df = pd.DataFrame(results)[metric_cols]

print('\n' + '═'*80)
print('PERFORMANS KIYASLAMA TABLOSU')
print('═'*80)
print(comparison_df.to_string(index=False))
print('═'*80)

## 🏆 6. ŞAMPİYON STRATEJİ İLANI

In [ ]:
def declare_champion(results: list) -> dict:
    """Bileşik skor ile şampiyon stratejiyi belirle."""
    scores = []
    for r in results:
        # Normalize edilmiş bileşik skor (her metrik eşit ağırlıklı)
        ret_score    = r['Toplam Getiri %']
        sharpe_score = r['Sharpe'] * 20          # ölçek normalize
        dd_score     = -r['Max Drawdown %']      # pozitife çevir
        wr_score     = r['Win Rate %'] if not pd.isna(r['Win Rate %']) else 0
        composite    = 0.35*ret_score + 0.30*sharpe_score + 0.20*dd_score + 0.15*wr_score
        scores.append((r['Strateji'], composite, r))

    champion = max(scores, key=lambda x: x[1])

    print('\n' + '🏆'*20)
    print(f'  ŞAMPİYON STRATEJİ: {champion[0]}')
    print('🏆'*20)
    r = champion[2]
    print(f"  Toplam Getiri : {r['Toplam Getiri %']:+.2f}%")
    print(f"  Sharpe Oranı  : {r['Sharpe']:.3f}")
    print(f"  Max Drawdown  : {r['Max Drawdown %']:.2f}%")
    print(f"  Win Rate      : {r['Win Rate %']:.2f}%")
    print(f"  İşlem Sayısı  : {r['İşlem Sayısı']}")
    print(f"  Bileşik Skor  : {champion[1]:.2f}")

    # Sıralama
    print('\n  SIRALAMA:')
    for rank, (name, score, _) in enumerate(sorted(scores, key=lambda x: x[1], reverse=True), 1):
        print(f'    {rank}. {name}  (skor: {score:.2f})')

    return champion[2]


champion = declare_champion(results)

## 🧠 7. KENDİNİ GELİŞTİREN MOTOR — Walk-Forward + Optuna Otomatik Ayar

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  HAFIZA MOTORU — Geçmiş performansı yükle / kaydet
# ══════════════════════════════════════════════════════════════════

def load_history() -> list:
    if HISTORY_FILE.exists():
        with open(HISTORY_FILE) as f:
            return json.load(f)
    return []

def save_history(history: list):
    with open(HISTORY_FILE, 'w') as f:
        json.dump(history, f, indent=2, default=str)

def load_best_params() -> dict:
    if PARAMS_FILE.exists():
        with open(PARAMS_FILE) as f:
            return json.load(f)
    return {}

def save_best_params(params: dict):
    with open(PARAMS_FILE, 'w') as f:
        json.dump(params, f, indent=2)

def performance_trend(history: list, last_n: int = 5) -> str:
    """Son N çalıştırmanın win rate trendini yorumla."""
    if len(history) < 2:
        return 'Henüz yeterli geçmiş yok'
    recent = [h.get('xgb_winrate', 50) for h in history[-last_n:]]
    trend  = recent[-1] - recent[0]
    if trend > 3:
        return f'📈 Geliştiriliyor (+{trend:.1f}% son {len(recent)} turda)'
    elif trend < -3:
        return f'📉 Gerileme var ({trend:.1f}%), yeniden optimizasyon önerilir'
    else:
        return f'➡️  Stabil ({recent[-1]:.1f}% win rate)'


history = load_history()
prev_params = load_best_params()

print(f'Geçmiş çalıştırma sayısı : {len(history)}')
print(f'Performans trendi        : {performance_trend(history)}')
if prev_params:
    print(f'Önceki en iyi parametreler yüklendi: {list(prev_params.keys())}')


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  WALK-FORWARD DOĞRULAMA — Model gerçekten öğreniyor mu?
#  Veriyi 5 eşit pencereye böler, her seferinde birikimli eğitir,
#  bir sonraki pencerede test eder. Zaman yanlılığını ortadan kaldırır.
# ══════════════════════════════════════════════════════════════════

def walk_forward_validation(df: pd.DataFrame,
                             n_splits: int = 5,
                             feature_cols: list = None) -> pd.DataFrame:
    avail = [c for c in (feature_cols or FEATURE_COLS) if c in df.columns]
    dfc   = df[avail + ['Label']].dropna().copy()
    n     = len(dfc)

    fold_size = n // (n_splits + 1)
    records   = []

    print(f'Walk-Forward: {n_splits} tur | Her tur ≈ {fold_size} bar')
    print('─' * 55)

    for fold in range(n_splits):
        train_end  = fold_size * (fold + 1)
        test_start = train_end
        test_end   = min(train_end + fold_size, n)

        X_train = dfc[avail].iloc[:train_end]
        y_train = dfc['Label'].iloc[:train_end]
        X_test  = dfc[avail].iloc[test_start:test_end]
        y_test  = dfc['Label'].iloc[test_start:test_end]

        if len(X_test) == 0:
            continue

        sc = StandardScaler()
        X_tr_sc = sc.fit_transform(X_train)
        X_te_sc = sc.transform(X_test)

        pw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        mdl = XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=pw, use_label_encoder=False,
            eval_metric='logloss', random_state=42, n_jobs=-1
        )
        mdl.fit(X_tr_sc, y_train, verbose=False)
        preds = mdl.predict(X_te_sc)
        acc   = accuracy_score(y_test, preds) * 100

        date_start = dfc.index[test_start].date()
        date_end   = dfc.index[test_end - 1].date()

        records.append({
            'Tur'         : f'Tur {fold+1}',
            'Eğitim Barı' : train_end,
            'Test Dönemi' : f'{date_start} → {date_end}',
            'Doğruluk %'  : round(acc, 2),
            'Trend'       : '✅ İyi' if acc >= 55 else ('⚠️ Orta' if acc >= 50 else '❌ Zayıf'),
        })
        print(f'  Tur {fold+1}: {date_start} → {date_end} | Doğruluk: {acc:.1f}% {records[-1]["Trend"]}')

    wf_df  = pd.DataFrame(records)
    avg_acc = wf_df['Doğruluk %'].mean()
    trend   = wf_df['Doğruluk %'].iloc[-1] - wf_df['Doğruluk %'].iloc[0]

    print('─' * 55)
    print(f'Ortalama Doğruluk : {avg_acc:.2f}%')
    print(f'İlk→Son Tur Farkı : {trend:+.2f}% '
          + ('(Model zamanla güçleniyor 📈)' if trend > 0 else '(Stabil kalıyor ➡️)'))
    return wf_df


wf_results = walk_forward_validation(df)
wf_results


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  OTOMATİK PARAMETRE OPTİMİZASYONU (Optuna)
#  Her çalıştırmada XGBoost hiperparametrelerini ve RSI/MACD eşiklerini
#  otomatik olarak dener ve en iyisini hafızaya yazar.
#
#  NASIL ÇALIŞIR?
#  Optuna, "acı çekerek öğrenen" bir arama algoritmasıdır.
#  50 farklı parametre kombinasyonunu dener, en yüksek win rate'i
#  veren kombinasyonu seçer ve bunu bir sonraki çalıştırma için saklar.
# ══════════════════════════════════════════════════════════════════

def optuna_optimize(df: pd.DataFrame, n_trials: int = 50) -> dict:
    avail = [c for c in FEATURE_COLS if c in df.columns]
    dfc   = df[avail + ['Label']].dropna().copy()
    split = int(len(dfc) * 0.8)
    X_tr, y_tr = dfc[avail].iloc[:split], dfc['Label'].iloc[:split]
    X_te, y_te = dfc[avail].iloc[split:], dfc['Label'].iloc[split:]

    sc         = StandardScaler()
    X_tr_sc    = sc.fit_transform(X_tr)
    X_te_sc    = sc.transform(X_te)

    def objective(trial):
        params = dict(
            n_estimators    = trial.suggest_int('n_estimators', 100, 600),
            max_depth       = trial.suggest_int('max_depth', 2, 7),
            learning_rate   = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            subsample       = trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree= trial.suggest_float('colsample_bytree', 0.5, 1.0),
            min_child_weight= trial.suggest_int('min_child_weight', 1, 10),
            gamma           = trial.suggest_float('gamma', 0, 1),
            reg_alpha       = trial.suggest_float('reg_alpha', 0, 1),
            reg_lambda      = trial.suggest_float('reg_lambda', 0.5, 2),
        )
        pw  = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
        mdl = XGBClassifier(
            **params, scale_pos_weight=pw,
            use_label_encoder=False, eval_metric='logloss',
            random_state=42, n_jobs=-1
        )
        mdl.fit(X_tr_sc, y_tr, verbose=False)
        return accuracy_score(y_te, mdl.predict(X_te_sc))

    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))

    # Önceki en iyi parametreleri başlangıç noktası olarak ekle
    prev = load_best_params()
    if prev:
        study.enqueue_trial(prev)

    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = study.best_params
    best['achieved_accuracy'] = round(study.best_value * 100, 2)
    best['optimized_at']      = datetime.now().isoformat()
    best['ticker']            = TICKER

    save_best_params(best)

    print(f'Optuna optimizasyonu tamamlandı ({n_trials} deneme)')
    print(f'En iyi doğruluk  : {best["achieved_accuracy"]}%')
    print(f'En iyi parametreler:')
    for k, v in best.items():
        if k not in ('achieved_accuracy', 'optimized_at', 'ticker'):
            print(f'  {k:<22}: {v}')

    return best


# Kaç trial deneyelim? (daha fazla = daha iyi ama daha yavaş)
OPTUNA_TRIALS = 50

# Geçmiş performans düştüyse veya ilk çalıştırmaysa optimize et
need_optim = (
    len(history) == 0
    or not prev_params
    or (len(history) >= 3 and
        history[-1].get('xgb_winrate', 50) < history[-2].get('xgb_winrate', 50) - 2)
)

if need_optim:
    print('Otomatik optimizasyon başlatılıyor...')
    best_xgb_params = optuna_optimize(df, n_trials=OPTUNA_TRIALS)
else:
    best_xgb_params = prev_params
    print(f'Mevcut parametreler yeterince iyi, optimizasyon atlandı.')
    print(f'(Önceki doğruluk: {prev_params.get("achieved_accuracy", "?")}%)')


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  OPTİMİZE EDİLMİŞ XGBoost MODELİNİ YENİDEN EĞİT & KAYDET
#  Optuna'nın bulduğu en iyi parametrelerle modeli yeniden kur,
#  diske kaydet. Bir sonraki çalıştırmada yüklenir → sıfırdan başlamaz.
# ══════════════════════════════════════════════════════════════════

def retrain_with_best_params(df: pd.DataFrame, best_params: dict):
    avail  = [c for c in FEATURE_COLS if c in df.columns]
    dfc    = df[avail + ['Label']].dropna().copy()
    split  = int(len(dfc) * TRAIN_RATIO)

    X_tr, y_tr = dfc[avail].iloc[:split], dfc['Label'].iloc[:split]
    X_te, y_te = dfc[avail].iloc[split:], dfc['Label'].iloc[split:]

    sc         = StandardScaler()
    X_tr_sc    = sc.fit_transform(X_tr)
    X_te_sc    = sc.transform(X_te)

    # Optuna meta-alanlarını temizle
    mdl_params = {k: v for k, v in best_params.items()
                  if k not in ('achieved_accuracy', 'optimized_at', 'ticker')}

    pw  = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    mdl = XGBClassifier(
        **mdl_params, scale_pos_weight=pw,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, n_jobs=-1
    )
    mdl.fit(X_tr_sc, y_tr, verbose=False)
    acc = accuracy_score(y_te, mdl.predict(X_te_sc)) * 100

    # Diske kaydet
    joblib.dump(mdl, MODEL_FILE)
    joblib.dump(sc,  SCALER_FILE)
    print(f'Model kaydedildi → {MODEL_FILE}')
    print(f'Scaler kaydedildi → {SCALER_FILE}')
    print(f'Test doğruluğu (optimize edilmiş): {acc:.2f}%')

    return mdl, sc, dfc.index[split], acc


xgb_model_opt, xgb_scaler_opt, test_start_opt, final_acc = retrain_with_best_params(df, best_xgb_params)

# Optimize edilmiş modelle sinyal üret
avail_cols = [c for c in FEATURE_COLS if c in df.columns]
dfc_full   = df[avail_cols + ['Label']].dropna()
split_n    = int(len(dfc_full) * TRAIN_RATIO)
X_test_all = dfc_full[avail_cols].iloc[split_n:]
X_te_sc    = xgb_scaler_opt.transform(X_test_all)
preds_opt  = xgb_model_opt.predict(X_te_sc)

sig_C_opt  = pd.Series(0, index=df.index)
sig_C_opt.loc[X_test_all.index] = np.where(preds_opt == 1, 1, -1)

# Backtest ile karşılaştır
res_C_opt = run_backtest(df, sig_C_opt, 'C-Optuna: XGBoost+Optuna', start_date=test_start_opt)
print(f"\n  Getiri: {res_C_opt['Toplam Getiri %']:+.2f}% | "
      f"Sharpe: {res_C_opt['Sharpe']:.3f} | "
      f"Win Rate: {res_C_opt['Win Rate %']:.2f}%")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PERFORMANS HAFIZASINI GÜNCELLE
#  Her çalıştırmada sonuçları JSON'a ekle.
#  Böylece model kendini kaç çalıştırmadır takip edebilir.
# ══════════════════════════════════════════════════════════════════

all_results = [res_A, res_B, res_C, res_C_opt]
champ_full  = declare_champion(all_results)

run_record = {
    'tarih'           : datetime.now().isoformat(),
    'ticker'          : TICKER,
    'interval'        : INTERVAL_STR,
    'ta_return'       : res_A['Toplam Getiri %'],
    'ta_winrate'      : res_A['Win Rate %'],
    'garch_return'    : res_B['Toplam Getiri %'],
    'garch_winrate'   : res_B['Win Rate %'],
    'xgb_return'      : res_C['Toplam Getiri %'],
    'xgb_winrate'     : res_C['Win Rate %'],
    'xgb_opt_return'  : res_C_opt['Toplam Getiri %'],
    'xgb_opt_winrate' : res_C_opt['Win Rate %'],
    'champion'        : champ_full['Strateji'],
    'wf_avg_accuracy' : round(wf_results['Doğruluk %'].mean(), 2),
    'optuna_accuracy' : final_acc,
}

history.append(run_record)
save_history(history)
print(f'\nPerformans kaydedildi. Toplam geçmiş: {len(history)} çalıştırma.')
print(f'Şampiyon bu turda: {champ_full["Strateji"]}')


## 📈 7. GÖRSELLEŞTİRME

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14), facecolor='#0d1117')
fig.suptitle(f'{TICKER} — Strateji Karşılaştırması (Test Dönemi)', 
             fontsize=16, color='white', fontweight='bold', y=0.98)

# Panel 1: Kümülatif Getiri
ax1 = axes[0]
ax1.set_facecolor('#0d1117')
for i, r in enumerate(results):
    cum = r['_data']['Cum_Return']
    ax1.plot(cum.index, cum.values, label=f"{r['Strateji']} ({r['Toplam Getiri %']:+.1f}%)",
             color=COLORS[i], linewidth=1.8)
# Buy & Hold
bh = results[0]['_data']['BH_Return']
ax1.plot(bh.index, bh.values, '--', color='gray', alpha=0.6,
         label=f"Buy & Hold ({results[0]['BH Getiri %']:+.1f}%)")
ax1.set_ylabel('Kümülatif Getiri (1 = başlangıç)', color='white')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(alpha=0.15)
ax1.tick_params(colors='white')
ax1.spines[['top','right','left','bottom']].set_color('#333')

# Panel 2: Drawdown
ax2 = axes[1]
ax2.set_facecolor('#0d1117')
for i, r in enumerate(results):
    cum = r['_data']['Cum_Return']
    roll_max = cum.cummax()
    dd = (cum / roll_max - 1) * 100
    ax2.fill_between(dd.index, dd.values, 0, alpha=0.4, color=COLORS[i],
                     label=f"{r['Strateji']} (MaxDD: {r['Max Drawdown %']:.1f}%)")
ax2.set_ylabel('Drawdown (%)', color='white')
ax2.legend(loc='lower left', fontsize=9)
ax2.grid(alpha=0.15)
ax2.tick_params(colors='white')
ax2.spines[['top','right','left','bottom']].set_color('#333')

# Panel 3: Fiyat + Sinyaller (XGBoost)
ax3 = axes[2]
ax3.set_facecolor('#0d1117')
price_test = df.loc[df.index >= test_start, 'Close']
ax3.plot(price_test.index, price_test.values, color='#aaaaaa', linewidth=1, label='Fiyat')

# Al sinyalleri
buy_mask  = (sig_C == 1)  & (df.index >= test_start)
sell_mask = (sig_C == -1) & (df.index >= test_start)
ax3.scatter(df.index[buy_mask],  df.loc[buy_mask, 'Close'],
            marker='^', color='#00ff88', s=40, label='XGBoost Al', zorder=5, alpha=0.8)
ax3.scatter(df.index[sell_mask], df.loc[sell_mask, 'Close'],
            marker='v', color='#ff6b6b', s=40, label='XGBoost Sat', zorder=5, alpha=0.8)
ax3.set_ylabel('Fiyat (TL)', color='white')
ax3.legend(loc='upper left', fontsize=9)
ax3.grid(alpha=0.15)
ax3.tick_params(colors='white')
ax3.spines[['top','right','left','bottom']].set_color('#333')

for ax in axes:
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_color('white')

plt.tight_layout()
plt.savefig('strategy_comparison.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Grafik kaydedildi: strategy_comparison.png')

In [ ]:
# XGBoost Özellik Önem Grafiği
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0d1117')

# Sol: Feature importance
top20 = feat_imp.head(20)
bars  = ax1.barh(top20.index[::-1], top20.values[::-1], color='#4ecdc4', alpha=0.8)
ax1.set_xlabel('Önem Skoru', color='white')
ax1.set_title('XGBoost — Top 20 Özellik', color='white', fontweight='bold')
ax1.set_facecolor('#0d1117')
ax1.tick_params(colors='white')
ax1.spines[['top','right','left','bottom']].set_color('#333')
for label in ax1.get_xticklabels() + ax1.get_yticklabels():
    label.set_color('white')

# Sağ: Metrik karşılaştırma (radar benzeri bar)
metrics  = ['Toplam Getiri %', 'Sharpe', 'Win Rate %']
names    = [r['Strateji'].split(':')[0] for r in results]
x        = np.arange(len(metrics))
width    = 0.25

for i, r in enumerate(results):
    vals = [r['Toplam Getiri %'], r['Sharpe']*10, r['Win Rate %']]
    ax2.bar(x + i*width, vals, width, label=r['Strateji'], color=COLORS[i], alpha=0.8)

ax2.set_xticks(x + width)
ax2.set_xticklabels(['Getiri %', 'Sharpe×10', 'Win Rate %'], color='white')
ax2.set_title('Metrik Karşılaştırması', color='white', fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_facecolor('#0d1117')
ax2.tick_params(colors='white')
ax2.spines[['top','right','left','bottom']].set_color('#333')
ax2.axhline(0, color='white', linewidth=0.5)
for label in ax2.get_xticklabels() + ax2.get_yticklabels():
    label.set_color('white')

plt.suptitle(f'{TICKER} — Model Analizi', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## ⚡ 8. CANLI SİNYAL ÜRETİCİ — Şu Anki Bar İçin Tahmin

In [ ]:
def generate_live_signal(ticker: str, exchange: str,
                         index_ticker: str, index_exchange: str,
                         interval: Interval, interval_str: str,
                         model, scaler, feature_cols: list) -> dict:
    """
    Gerçek zamanlı (veya en güncel) bar için tüm stratejilerin sinyalini üret.
    """
    print(f'\n{"═"*60}')
    print(f'CANLI SİNYAL — {ticker} | {interval_str} | {datetime.now().strftime("%Y-%m-%d %H:%M")}')
    print('═'*60)

    live_df = download_data(ticker, exchange, index_ticker, index_exchange,
                            interval, n_bars=500)
    live_df = build_features(live_df)

    last = live_df.dropna().iloc[-1]
    prev = live_df.dropna().iloc[-2]

    # ── Strateji A Sinyali ──────────────────────────────────────
    macd_cross = (last['MACD_Hist'] > 0) and (prev['MACD_Hist'] <= 0)
    rsi_ok     = 30 < last['RSI'] < 70
    trend_up   = last['EMA9'] > last['EMA21']
    vol_up     = last['Vol_Ratio'] > 1.0

    if macd_cross and rsi_ok and trend_up and vol_up:
        sig_a = 'AL  ▲'
    elif (last['MACD_Hist'] < 0 and prev['MACD_Hist'] >= 0) or (last['EMA9'] < last['EMA21']):
        sig_a = 'SAT ▼'
    else:
        sig_a = 'BEKLE ─'

    # ── Strateji C (XGBoost) Sinyali ───────────────────────────
    avail = [c for c in feature_cols if c in live_df.columns]
    last_features = live_df[avail].dropna().iloc[-1:]
    if len(last_features) > 0:
        last_sc  = scaler.transform(last_features)
        pred     = model.predict(last_sc)[0]
        proba    = model.predict_proba(last_sc)[0]
        sig_c    = f'AL  ▲ (güven: {proba[1]:.1%})' if pred == 1 else f'SAT ▼ (güven: {proba[0]:.1%})'
    else:
        sig_c = 'Yeterli veri yok'

    # ── Özet Çıktı ─────────────────────────────────────────────
    print(f'\nFiyat       : {last["Close"]:.2f} TL')
    print(f'RSI-14      : {last["RSI"]:.1f}')
    print(f'MACD Hist   : {last["MACD_Hist"]:.4f}')
    print(f'EMA9/21     : {last["EMA9"]:.2f} / {last["EMA21"]:.2f}')
    print(f'ATR %       : {last["ATR_Pct"]:.3%}')
    print(f'Hacim Oranı : {last["Vol_Ratio"]:.2f}x')
    print(f'ADX         : {last["ADX"]:.1f}')
    print(f'BB Pozisyon : {last["BB_Pct"]:.1%}')
    print()
    print(f'Strateji A (Klasik TA) : {sig_a}')
    print(f'Strateji C (XGBoost)   : {sig_c}')

    atr    = last['ATR']
    close  = last['Close']
    stop   = round(close - 1.5 * atr, 2)
    target = round(close + 2.5 * atr, 2)
    rr     = round((target - close) / (close - stop), 2) if close > stop else 'N/A'
    print(f'\n[SCALP PARAMETRELERİ]')
    print(f'  Giriş       : {close:.2f} TL')
    print(f'  Stop Loss   : {stop:.2f} TL  (-{(close-stop)/close:.1%})')
    print(f'  Hedef       : {target:.2f} TL  (+{(target-close)/close:.1%})')
    print(f'  Risk/Ödül   : 1:{rr}')

    return {'sig_a': sig_a, 'sig_c': sig_c, 'close': close,
            'stop': stop, 'target': target, 'rr': rr}


# Canlı sinyal üret
live = generate_live_signal(
    TICKER, EXCHANGE, INDEX_TKR, INDEX_EXCHANGE,
    TV_INTERVAL, INTERVAL_STR,
    xgb_model, xgb_scaler, FEATURE_COLS
)


## 🔄 11. GERİ BESLEME DÖNGÜSÜ — Öğrenilenleri Koda Uygula

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  GERİ BESLEME DÖNGÜSÜ
#
#  NASIL ÇALIŞIR?
#  1. Tarayıcı her hisse için 4 motorun alt skorlarını kaydeder.
#  2. N bar sonra aynı hisselerin gerçek getirisini ölçer.
#  3. "Hangi motor o an için en öngörücüydü?" sorusunu yanıtlar.
#  4. Her motorun ağırlığını son 30 günün performansına göre günceller.
#  5. Güncellenen ağırlıklar PARAMS_FILE'a yazılır, bir sonraki
#     taramada otomatik olarak kullanılır.
#
#  YANİ: Kod önce tahmin eder, sonra yanılıp yanılmadığını ölçer,
#  yanılan motoru cezalandırır, isabetli motoru ödüllendirir.
# ══════════════════════════════════════════════════════════════════

SIGNAL_LOG_FILE  = MEMORY_DIR / 'signal_log.json'
WEIGHTS_FILE     = MEMORY_DIR / 'motor_weights.json'

# Varsayılan ağırlıklar — geri besleme yokken eşit başlar
DEFAULT_WEIGHTS = {'ta': 0.35, 'xgb': 0.35, 'garch': 0.15, 'pattern': 0.15}


def load_signal_log() -> list:
    if SIGNAL_LOG_FILE.exists():
        with open(SIGNAL_LOG_FILE) as f:
            return json.load(f)
    return []


def save_signal_log(log: list):
    with open(SIGNAL_LOG_FILE, 'w') as f:
        json.dump(log, f, indent=2, default=str)


def load_motor_weights() -> dict:
    if WEIGHTS_FILE.exists():
        with open(WEIGHTS_FILE) as f:
            w = json.load(f)
        # Tüm anahtarlar mevcut mu kontrol et
        if all(k in w for k in DEFAULT_WEIGHTS):
            return w
    return DEFAULT_WEIGHTS.copy()


def save_motor_weights(weights: dict):
    with open(WEIGHTS_FILE, 'w') as f:
        json.dump(weights, f, indent=2)


def log_scan_predictions(scan_df: pd.DataFrame, interval: str) -> list:
    """
    Tarama sonuçlarını her motorun alt skoruyla birlikte kaydet.
    N bar sonra gerçek getiriyle karşılaştırılacak.
    """
    horizon_map = {'1d': 5, '1h': 8, '15m': 12, '5m': 16, '1W': 2, '1M': 1}
    horizon     = horizon_map.get(interval, 5)

    new_entries = []
    for _, row in scan_df.iterrows():
        entry = {
            'tarih'     : datetime.now().isoformat(),
            'ticker'    : row['Hisse'],
            'fiyat'     : row['Fiyat'],
            'horizon'   : horizon,
            'ta_score'  : row.get('TA', 0),
            'xgb_prob'  : row.get('YZ_%', 50) / 100,
            'garch_dir' : 1  if 'Düşük' in str(row.get('GARCH','')) else
                          (-1 if 'Yüksek' in str(row.get('GARCH','')) else 0),
            'pat_ret'   : float(str(row.get('Patern','0%')).replace('%','') or 0) / 100,
            'composite' : row.get('Skor', 5),
            'sinyal'    : row.get('Sinyal', ''),
            'gercek_ret': None,   # doldurulacak
            'dogrulandi': False,
        }
        new_entries.append(entry)

    log = load_signal_log()
    log.extend(new_entries)
    save_signal_log(log)
    print(f'{len(new_entries)} sinyal kayıt altına alındı. '
          f'Toplam kayıt: {len(log)}')
    return new_entries


def validate_past_predictions(tv_interval: Interval) -> pd.DataFrame:
    """
    Daha önce kaydedilen sinyallerin gerçek getirisini tvdatafeed ile çek.
    Yalnızca henüz doğrulanmamış ve yeterli süre geçmiş kayıtları işle.
    """
    log = load_signal_log()
    if not log:
        print('Henüz doğrulanacak sinyal yok.')
        return pd.DataFrame()

    tv       = get_tv_client()
    updated  = 0
    results  = []

    for entry in log:
        if entry.get('dogrulandi'):
            results.append(entry)
            continue

        # Yeterli süre geçti mi?
        tarih  = datetime.fromisoformat(entry['tarih'])
        gecen  = (datetime.now() - tarih).total_seconds() / 3600  # saat

        min_bekleme = {'1d':24,'1h':8,'15m':2,'5m':1,'1W':168,'1M':720}
        interval_str_map = {
            Interval.in_daily:'1d', Interval.in_1_hour:'1h',
            Interval.in_15_minute:'15m', Interval.in_5_minute:'5m',
        }
        iv_str = interval_str_map.get(tv_interval, '1d')
        if gecen < min_bekleme.get(iv_str, 24):
            results.append(entry)
            continue

        # Gerçek getiriyi çek
        try:
            raw = tv.get_hist(symbol=entry['ticker'], exchange='BIST',
                              interval=tv_interval, n_bars=50)
            if raw is not None and not raw.empty:
                closes     = tv_to_df(raw)['Close'].values
                giri_fiyat = entry['fiyat']
                # Giriş fiyatına en yakın bardan itibaren horizon bar ilerisi
                idx = np.argmin(np.abs(closes - giri_fiyat))
                idx_hedef  = min(idx + entry['horizon'], len(closes) - 1)
                ret        = (closes[idx_hedef] - closes[idx]) / closes[idx]
                entry['gercek_ret']   = round(ret, 5)
                entry['dogrulandi']   = True
                updated += 1
        except Exception:
            pass

        results.append(entry)

    save_signal_log(results)
    validated = [r for r in results if r.get('dogrulandi')]
    print(f'{updated} yeni sinyal doğrulandı. '
          f'Toplam doğrulanmış: {len(validated)}')

    if validated:
        return pd.DataFrame(validated)
    return pd.DataFrame()


def update_motor_weights(validated_df: pd.DataFrame,
                         min_records: int = 20) -> dict:
    """
    Hangi motor gerçek getiriyi daha iyi öngördü?
    Her motorun tahmini ile gerçek yön arasındaki korelasyona göre
    ağırlıkları güncelle. Softmax normalizasyon ile 0-1 arasına çek.

    Minimum kayıt sayısına ulaşılmamışsa mevcut ağırlıkları koru.
    """
    current = load_motor_weights()

    if len(validated_df) < min_records:
        print(f'Henüz yeterli veri yok ({len(validated_df)}/{min_records} kayıt). '
              f'Mevcut ağırlıklar korunuyor.')
        return current

    df = validated_df.copy()
    df['gercek_yon'] = np.sign(df['gercek_ret'])

    # Her motorun "yön isabeti" → 1=doğru, 0=yanlış
    df['ta_hit']   = ((df['ta_score'] / 9 - 0.5) * df['gercek_yon'] > 0).astype(float)
    df['xgb_hit']  = ((df['xgb_prob'] - 0.5)     * df['gercek_yon'] > 0).astype(float)
    df['garch_hit']= (df['garch_dir']             * df['gercek_yon'] > 0).astype(float)
    df['pat_hit']  = (df['pat_ret']               * df['gercek_yon'] > 0).astype(float)

    # Son 30 kayıt üzerinden isabetleri hesapla
    recent = df.tail(30)
    raw_scores = {
        'ta'     : recent['ta_hit'].mean(),
        'xgb'    : recent['xgb_hit'].mean(),
        'garch'  : recent['garch_hit'].mean(),
        'pattern': recent['pat_hit'].mean(),
    }

    # Softmax ile ağırlıklandır (daha isabetli motor daha fazla ağırlık alır)
    vals    = np.array(list(raw_scores.values()))
    exp_v   = np.exp((vals - vals.mean()) * 5)   # sıcaklık parametresi=5
    weights = exp_v / exp_v.sum()
    new_w   = dict(zip(raw_scores.keys(), weights.round(4).tolist()))

    # Ani sapmayı önle: eski ağırlıkla %70-30 harmanla
    blended = {k: round(0.70 * new_w[k] + 0.30 * current[k], 4)
               for k in current}
    # Normalize: toplam 1 olsun
    total = sum(blended.values())
    blended = {k: round(v/total, 4) for k, v in blended.items()}

    save_motor_weights(blended)

    print('\n' + '─'*55)
    print('MOTOR AĞIRLIKLARI GÜNCELLENDİ:')
    print(f'  {"Motor":<10} {"Önceki":>8} {"İsabet%":>9} {"Yeni":>8} {"Değişim":>9}')
    print('  ' + '─'*50)
    for k in current:
        old_w  = current[k]
        new_ww = blended[k]
        hit    = raw_scores[k] * 100
        chg    = new_ww - old_w
        arrow  = '▲' if chg > 0.005 else ('▼' if chg < -0.005 else '─')
        print(f'  {k:<10} {old_w:>8.4f} {hit:>8.1f}% {new_ww:>8.4f} {chg:>+8.4f} {arrow}')

    print('\n  Yorum:')
    best_motor  = max(raw_scores, key=raw_scores.get)
    worst_motor = min(raw_scores, key=raw_scores.get)
    print(f'  En isabetli motor : {best_motor} ({raw_scores[best_motor]*100:.1f}% doğru)')
    print(f'  En yanılgılı motor: {worst_motor} ({raw_scores[worst_motor]*100:.1f}% doğru)')
    print(f'  Ağırlıklar bir sonraki taramada otomatik olarak uygulanır.')

    return blended


# ══════════════════════════════════════════════════════════════════
#  DÖNGÜYÜ ÇALIŞTIR
# ══════════════════════════════════════════════════════════════════

# Adım 1: Geçmiş tahminleri doğrula
print('ADIM 1 — Geçmiş tahminlerin gerçek getirisi ölçülüyor...')
validated_df = validate_past_predictions(TV_INTERVAL)

# Adım 2: Motor ağırlıklarını güncelle
print('\nADIM 2 — Motor ağırlıkları güncelleniyor...')
updated_weights = update_motor_weights(validated_df)

# Adım 3: Güncel ağırlıkları tarayıcı için global değişkene aktar
MOTOR_WEIGHTS = updated_weights
print(f'\nGüncel ağırlıklar: {MOTOR_WEIGHTS}')

# Adım 4: Tarayıcı bu ağırlıkları kullandığını bilsin
print('\nBu ağırlıklar bir sonraki scan_all_bist_full() çağrısında')
print('_composite_score() içinde otomatik olarak kullanılacak.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  _composite_score() FONKSİYONUNU ÖĞRENILMIŞ AĞIRLIKLARA GÜNCELLE
#  Bu hücre scanner'dan önce çalışmalı.
# ══════════════════════════════════════════════════════════════════

# Öğrenilmiş ağırlıkları yükle (yoksa varsayılan)
_motor_w = load_motor_weights()

def _composite_score(ta_s, xgb_p, garch_s, pat_s,
                     weights: dict = None) -> float:
    """
    Ağırlıklı bileşik skor — ağırlıklar geri besleme döngüsünden gelir.
    Her çalıştırmada hangi motor geçmişte daha isabetliyse o daha fazla
    oy hakkı kazanır.
    """
    w = weights or _motor_w
    ta_norm    = ta_s / 9.0
    xgb_norm   = xgb_p
    garch_norm = (garch_s + 1) / 2
    pat_norm   = (pat_s  + 1) / 2

    composite = (w['ta']      * ta_norm   +
                 w['xgb']     * xgb_norm  +
                 w['garch']   * garch_norm +
                 w['pattern'] * pat_norm) * 10
    return round(composite, 2)


print(f'Adaptif _composite_score() aktif.')
print(f'Güncel motor ağırlıkları:')
for k, v in _motor_w.items():
    bar = '█' * int(v * 40)
    print(f'  {k:<10}: {v:.4f}  {bar}')


## 🔍 9. BIST-100 TARAYICI — En İyi Fırsatları Bul

In [ ]:
import concurrent.futures, time

# ══════════════════════════════════════════════════════════════════
#  BIST TAM LİSTESİ (~494 hisse)
# ══════════════════════════════════════════════════════════════════

_RAW_SYMBOLS = [
    'AACTR','ACSEL','ADEL','ADESE','AEFES','AFYON','AGESA','AGROT','AHGAZ','AKBNK',
    'AKCNS','AKENR','AKFGY','AKFIN','AKGRT','AKMGY','AKSA','AKSEN','AKSGY','AKSUE',
    'AKTK','ALARK','ALBRK','ALCAR','ALFAS','ALGYO','ALKA','ALKIM','ALKLC','ALMAD',
    'ALVES','ANELE','ANGEN','ANHYT','ANSGR','ARASE','ARCLK','ARDYZ','ARENA','ARSAN',
    'ARTMS','ARZUM','ASELS','ASGYO','ASUZU','ATAGY','ATAKP','ATATP','ATEKS','ATLAS',
    'AVGYO','AVHOL','AVOD','AVTUR','AYCES','AYDEM','AYGAZ','AZTEK',
    'BAGFS','BAKAB','BALAT','BANVT','BARMA','BASCM','BASGZ','BAYRK','BERA','BEYAZ',
    'BFREN','BIMAS','BIOEN','BIONC','BIZIM','BJKAS','BLCYT','BMEKS','BNTAS','BOBET',
    'BORLS','BORSK','BOSSA','BRISA','BRKO','BRKSN','BRKVY','BRMEN','BRYAT','BSOKE',
    'BTCIM','BUCIM','BURCE','BURVA','BVSAN','BYDNR',
    'CANTE','CARFA','CASA','CBIGY','CCOLA','CELHA','CEMAS','CEMTS','CEOEM','CIMSA',
    'CLEBI','CLKHO','CMASA','CMBTN','CMENT','COKG','CRDFA','CRFSA','CUSAN',
    'DAGHL','DAGI','DAPGM','DARDL','DENGE','DENIZ','DERIM','DESA','DESPC','DEVA',
    'DGATE','DGKLB','DGZTE','DITAS','DMRGD','DMSAS','DNISI','DOAS','DOBUR','DOCO',
    'DOGUB','DOHOL','DOWAL','DTRND','DURAK','DYOBY','DZGYO',
    'EBEBK','ECILC','ECZYT','EDIP','EGGUB','EGPRO','EGSER','EKGYO','EKSUN','ELITE',
    'EMKEL','EMNIS','ENERY','ENKAI','ENSRI','ERCB','EREGL','ERSU','ESCAR','ESCOM',
    'ESEN','ETGR','ETYAT','EUHOL','EUREN','EUYO','EYGYO',
    'FADE','FENER','FMIZP','FONET','FORMT','FORTE','FRIGO','FROTO','FZLGY',
    'GARAN','GARFA','GEDIK','GEDZA','GENIL','GENTS','GEREL','GESAN','GILGZ','GLBMD',
    'GLCVY','GLYHO','GMTAS','GOKNR','GOLTS','GOODY','GOZDE','GRSEL','GRTHO','GRTRK',
    'GSDDE','GSDHO','GSRAY','GUBRF','GWIND','GZNMI',
    'HALKB','HATEK','HDFGS','HEDEF','HEKTS','HKTM','HLGYO','HOROZ','HTTBT','HUBVC',
    'HUNER','HURGZ',
    'ICBCT','IDGYO','IEYHO','IHAAS','IHEVA','IHGZT','IHLAS','IHLGM','IHYAY','IKGYO',
    'IMASM','INDES','INFO','INGRM','INTEM','INVEO','IPEKE','ISATR','ISBIR','ISCTR',
    'ISDMR','ISFIN','ISGSY','ISGYO','ISKPL','ISKUR','ISYAT','ITTFH','IZFAS','IZINV',
    'IZMDC','IZODC',
    'JANTS',
    'KAPLM','KARSN','KARTN','KARYE','KATMR','KAYSE','KBORU','KCAER','KCHOL','KENT',
    'KERVN','KERVT','KFEIN','KGYO','KHOLS','KILER','KLGYO','KLKIM','KLMSN','KLNMA',
    'KLRHO','KLSER','KMPUR','KNFRT','KONKA','KONTR','KONYA','KOPOL','KORDS','KOTON',
    'KOZAA','KOZAL','KRDMA','KRDMB','KRDMD','KRGYO','KRONT','KRPLS','KRSTL','KRTEK',
    'KRVGD','KSTUR','KTLEV','KUTPO','KUVVA',
    'LIDER','LIDFA','LILAK','LINK','LKMNH','LOGO','LRSHO','LUKSK',
    'MAALT','MACKO','MAGEN','MAKIM','MAKTK','MANAS','MARKA','MARTI','MAVI','MEDTR',
    'MEGAP','MEPET','MERCN','MERIT','MERKO','METRO','METUR','MGROS','MHRGY','MIPAZ',
    'MMCAS','MNDRS','MNDTR','MOBTL','MOGAN','MSGYO','MTRKS','MTRYO','MUTLU',
    'NATEN','NBDRD','NETAS','NIBAS','NTGAZ','NTTUR','NUGYO','NUHCM','NXGYO',
    'OBASE','ODAS','ODEYO','ODINE','OFSYM','ONCSM','ONRYT','ORCAY','ORGE','ORION',
    'ORKID','OSTIM','OTKAR','OYAKC','OYAYO','OYLUM','OZGYO','OZKGY',
    'PAGYO','PAMEL','PAPIL','PARSN','PEKGY','PENGD','PENTA','PETKM','PETUN','PGSUS',
    'PINSU','PKART','PKENT','PLTUR','PNLSN','POLHO','POLTK','POLYP','PRDGS','PRKAB',
    'PRKME','PRZMA','PSDTC','PTOFS',
    'QNBFB','QNBFL',
    'RALYH','RAYSG','RCIGY','RHEAG','RLYHO','RODRG','ROYAL','RTALB','RUBNS','RYGYO',
    'SAMAT','SANEL','SANFM','SANKO','SARKY','SASA','SAYAS','SDTTR','SEGYO','SEKFK',
    'SEKUR','SELEC','SELGD','SELVA','SEYKM','SILVR','SINET','SISE','SKBNK','SKYMD',
    'SMART','SNGYO','SNKRN','SODSN','SOKM','SRVGY','SUMAS','SUWEN',
    'TABGD','TATGD','TAVHL','TBORG','TCELL','TDGYO','TEKTU','TETMT','THYAO','TIRE',
    'TKFEN','TKNSA','TLMAN','TMPOL','TOASO','TOFAS','TOMFA','TPVGD','TRCAS','TRGYO',
    'TRILC','TRNSK','TSGYO','TTKOM','TTRAK','TUKAS','TUMAS','TUPRS','TUREX',
    'TURGG','TURGZ','TURSG','TSPOR','TTKAP',
    'UCAK','ULUFA','ULUSE','ULUUN','UMPAS','UNLU','USAK','USDMR','UTPYA',
    'VAKBN','VAKFN','VAKKO','VANGD','VBTYZ','VERUS','VESBE','VESTL','VKFYO','VKING',
    'VRGYO','WINTA',
    'YAPRK','YATAS','YAZIC','YBTAS','YDKYO','YEOTK','YESIL','YGGYO','YKBNK','YKSLN',
    'YONGA','YUNSA','YYLGD',
    'ZEDUR','ZOREN','ZRGYO',
]
_seen = set()
BIST_ALL = [(s, 'BIST') for s in _RAW_SYMBOLS if not (s in _seen or _seen.add(s))]
print(f'BIST tam listesi: {len(BIST_ALL)} hisse')


# ══════════════════════════════════════════════════════════════════
#  ÇOKLU MODEL TARAYICI
#  Her hisse için 4 bağımsız motor çalışır:
#   1. TA Skoru      — RSI/MACD/EMA/Hacim/ADX kuralları (0-9 puan)
#   2. XGBoost       — Ana hissede eğitilmiş model diğerlerine uygulanır
#                       (BIST'e özgü paternleri yakalar, transfer öğrenme)
#   3. GARCH-Lite    — Rolling volatilite yüzdelik dilimi (oynaklık rejimi)
#   4. Patern Benzer.— Son 20 barın geçmişle L2 mesafesi → beklenen yön
#  Ağırlıklı bileşik skor → nihai sıralama
# ══════════════════════════════════════════════════════════════════

# XGBoost ve scaler'ı yükle (önceki aşamada kaydedildi)
_xgb_model  = joblib.load(MODEL_FILE)  if MODEL_FILE.exists()  else None
_xgb_scaler = joblib.load(SCALER_FILE) if SCALER_FILE.exists() else None
_xgb_cols   = [c for c in FEATURE_COLS if c in df.columns]

# Referans patern: Ana hissenin son 20 barı (genel BIST trendi)
_ref_rets   = df['Log_Return'].dropna().values[-20:]
_ref_norm   = (_ref_rets - _ref_rets.mean()) / (_ref_rets.std() + 1e-9)


def _ta_score(last, prev) -> tuple:
    """Kural tabanlı TA puanı (0-9). Döner: (puan, etiket_listesi)"""
    score, labels = 0, []

    checks = [
        (last['RSI'] > 50,              '+RSI>50'),
        (last['MACD_Hist'] > 0,         '+MACD+'),
        (last['EMA9']  > last['EMA21'], '+EMA9>21'),
        (last['EMA21'] > last['SMA50'], '+EMA21>50'),
        (last['Vol_Ratio'] > 1.2,       '+Hacim'),
        (last['ADX'] > 25,              '+ADX'),
        (last['ROC5'] > 0,              '+ROC5'),
        (last['BB_Pct'] > 0.5,          '+BB'),
        (last['Stoch_K'] > last['Stoch_D'], '+Stoch'),
    ]
    for cond, lbl in checks:
        if cond:
            score += 1
            labels.append(lbl)

    return score, labels


def _xgb_score(d: pd.DataFrame) -> tuple:
    """
    XGBoost olasılık skoru.
    Modeli ana hissede eğittik; BIST paternleri benzer olduğundan
    diğer hisselere transfer edilebilir (özellikle normalize özelliklerle).
    Döner: (al_olasiligi 0-1, güven_etiketi)
    """
    if _xgb_model is None or _xgb_scaler is None:
        return 0.5, 'MODEL_YOK'
    try:
        avail = [c for c in _xgb_cols if c in d.columns]
        last_row = d[avail].dropna().iloc[-1:]
        if len(last_row) == 0:
            return 0.5, 'VERİ_YOK'
        proba = _xgb_model.predict_proba(_xgb_scaler.transform(last_row))[0][1]
        if proba >= 0.65:
            return proba, f'YZ:AL({proba:.0%})'
        elif proba >= 0.50:
            return proba, f'YZ:ZAYIFAL({proba:.0%})'
        elif proba >= 0.35:
            return proba, f'YZ:ZAYIFSAT({proba:.0%})'
        else:
            return proba, f'YZ:SAT({proba:.0%})'
    except Exception:
        return 0.5, 'YZ:HATA'


def _garch_lite_score(d: pd.DataFrame) -> tuple:
    """
    Hafif GARCH rejim tespiti: tam GARCH fit yerine rolling std yüzdelik dilimi.
    Tarama için yeterince güvenilir, 100x daha hızlı.
    Döner: (rejim_skoru -1/0/1, etiket)
      +1 = düşük oynaklık rejimi  → kırılım/fırsat bölgesi
       0 = orta oynaklık         → nötr
      -1 = yüksek oynaklık       → riskli, kaçın
    """
    try:
        rets   = d['Log_Return'].dropna()
        if len(rets) < 30:
            return 0, 'VOL:YETERSİZ'
        rol_std = rets.rolling(10).std()
        pct     = rol_std.rank(pct=True).iloc[-1]  # son noktanın yüzdelik dilimi
        if pct < 0.30:
            return 1,  f'GARCH:DüşükVol({pct:.0%})'   # sıkışma → fırsat
        elif pct > 0.70:
            return -1, f'GARCH:YüksekVol({pct:.0%})'  # kaos → kaçın
        else:
            return 0,  f'GARCH:OrtaVol({pct:.0%})'
    except Exception:
        return 0, 'VOL:HATA'


def _pattern_score(d: pd.DataFrame, lookback: int = 20) -> tuple:
    """
    Son 'lookback' barı referans paterniyle (ana hisse) karşılaştır.
    Benzer geçmiş desenler sonrasında ne olduğunu inceler.
    Döner: (yön_skoru -1/0/1, beklenen_getiri_tahmini, etiket)
    """
    try:
        rets = d['Log_Return'].dropna().values
        if len(rets) < lookback + 10:
            return 0, 0.0, 'PAT:YETERSİZ'

        window = rets[-lookback:]
        norm   = (window - window.mean()) / (window.std() + 1e-9)
        dist   = np.linalg.norm(_ref_norm - norm[:len(_ref_norm)])

        # Geçmişte benzer desenlerin ortalama sonraki 3-bar getirisini hesapla
        future_rets = []
        for i in range(lookback, len(rets) - lookback - 3):
            w  = rets[i-lookback:i]
            n  = (w - w.mean()) / (w.std() + 1e-9)
            d2 = np.linalg.norm(norm - n)
            if d2 < dist * 1.3:   # benzer desenler (eşik toleransı)
                future_rets.append(np.sum(rets[i:i+3]))

        if len(future_rets) < 3:
            return 0, 0.0, 'PAT:EŞLEŞMEYok'

        avg_fut = np.mean(future_rets)
        if avg_fut > 0.005:
            return 1,  avg_fut, f'PAT:Yukari({avg_fut*100:+.1f}%)'
        elif avg_fut < -0.005:
            return -1, avg_fut, f'PAT:Asagi({avg_fut*100:+.1f}%)'
        else:
            return 0,  avg_fut, f'PAT:Notr({avg_fut*100:+.1f}%)'
    except Exception:
        return 0, 0.0, 'PAT:HATA'


def _composite_score(ta_s, xgb_p, garch_s, pat_s) -> float:
    """
    Ağırlıklı bileşik skor (0-10 arası normalize).
    TA:35% + XGBoost:35% + GARCH:15% + Patern:15%
    """
    ta_norm    = ta_s / 9.0              # 0-1
    xgb_norm   = xgb_p                  # 0-1 (zaten olasılık)
    garch_norm = (garch_s + 1) / 2      # -1..1 → 0..1
    pat_norm   = (pat_s + 1) / 2        # -1..1 → 0..1

    composite  = (0.35 * ta_norm +
                  0.35 * xgb_norm +
                  0.15 * garch_norm +
                  0.15 * pat_norm) * 10  # 0-10 ölçeği
    return round(composite, 2)


def _scan_single_full(args) -> dict | None:
    """Her hisse için 4 modeli çalıştır ve bileşik skor hesapla."""
    tkr, exch, tv_interval, n_bars = args
    try:
        tv  = TvDatafeed()
        raw = tv.get_hist(symbol=tkr, exchange=exch,
                          interval=tv_interval, n_bars=n_bars)
        if raw is None or len(raw) < 60:
            return None

        d    = build_features(tv_to_df(raw).assign(Endeks_Close=lambda x: x['Close']))
        dna  = d.dropna()
        if len(dna) < 30:
            return None
        last = dna.iloc[-1]
        prev = dna.iloc[-2]

        # 4 motor
        ta_s,    ta_labels          = _ta_score(last, prev)
        xgb_p,   xgb_label         = _xgb_score(d)
        garch_s, garch_label        = _garch_lite_score(d)
        pat_s,   pat_ret, pat_label = _pattern_score(d)

        composite = _composite_score(ta_s, xgb_p, garch_s, pat_s)

        # Nihai sinyal
        if composite >= 7.0:
            signal = '🟢🟢 GÜÇLÜ AL'
        elif composite >= 5.5:
            signal = '🟢 AL'
        elif composite >= 4.5:
            signal = '🟡 BEKLE'
        elif composite >= 3.0:
            signal = '🔴 SAT'
        else:
            signal = '🔴🔴 GÜÇLÜ SAT'

        return {
            'Hisse'      : tkr,
            'Fiyat'      : round(last['Close'], 2),
            'Skor'       : composite,
            'Sinyal'     : signal,
            # Alt skorlar
            'TA'         : ta_s,
            'YZ_%'       : round(xgb_p * 100, 1),
            'GARCH'      : garch_label.split(':')[1] if ':' in garch_label else garch_label,
            'Patern'     : f'{pat_ret*100:+.2f}%',
            # Göstergeler
            'RSI'        : round(last['RSI'], 1),
            'ATR_%'      : round(last['ATR_Pct'] * 100, 2),
            'HacimOran'  : round(last['Vol_Ratio'], 2),
            'ADX'        : round(last['ADX'], 1),
            # Scalp seviyeleri
            'Stop'       : round(last['Close'] - 1.5 * last['ATR'], 2),
            'Hedef'      : round(last['Close'] + 2.5 * last['ATR'], 2),
        }
    except Exception:
        return None


def scan_all_bist_full(symbol_list, tv_interval=Interval.in_daily,
                       n_bars=300, max_workers=8, top_n=25):
    args_list = [(s, e, tv_interval, n_bars) for s, e in symbol_list]
    total, records, errors, done = len(args_list), [], 0, 0
    t0 = time.time()
    print(f'4-Model Tarama: {total} hisse | {max_workers} paralel iş parçacığı')
    print('Motorlar: TA Kuralları + XGBoost YZ + GARCH Rejim + Patern Benzerliği')
    print('─' * 65)

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_scan_single_full, a): a[0] for a in args_list}
        for future in concurrent.futures.as_completed(futures):
            done += 1
            res = future.result()
            if res:
                records.append(res)
            else:
                errors += 1
            if done % 75 == 0 or done == total:
                print(f'  {done}/{total} ({done/total*100:.0f}%) | '
                      f'Analiz: {len(records)} | Atlanan: {errors} | '
                      f'{time.time()-t0:.0f}s')

    full_df = pd.DataFrame(records).sort_values('Skor', ascending=False)
    print(f'\nTamamlandı: {len(records)} hisse ({time.time()-t0:.0f}s)')
    return full_df.head(top_n), full_df


# ─── TARAMAYI ÇALIŞTIR ───────────────────────────────────────────
FULL_SCAN     = True
SCAN_INTERVAL = Interval.in_daily  # güniçi için TV_INTERVAL

scan_list = BIST_ALL if FULL_SCAN else BIST_ALL[:100]

top_df, full_df = scan_all_bist_full(
    scan_list,
    tv_interval = SCAN_INTERVAL,
    n_bars      = 350,
    max_workers = 8,
    top_n       = 25,
)

# ─── SONUÇLARI GÖSTER ────────────────────────────────────────────
W2 = 85
print('\n' + '═' * W2)
print(f'  BIST 4-MODEL TARAMA — EN GÜÇLÜ 25 HİSSE  [{datetime.now().strftime("%d.%m.%Y %H:%M")}]')
print('═' * W2)
display_cols = ['Hisse','Fiyat','Skor','Sinyal','TA','YZ_%','GARCH','Patern',
                'RSI','HacimOran','ADX','Stop','Hedef']
print(top_df[display_cols].to_string(index=False))

print('\n' + '─' * W2)
print('SKOR TABLOSU (0-10):')
print('  Skor ≥ 7.0 → 🟢🟢 GÜÇLÜ AL  — 4 modelin büyük çoğunluğu alış işaret ediyor')
print('  Skor ≥ 5.5 → 🟢   AL         — Çoğunluk alış, girişe değer')
print('  Skor ≥ 4.5 → 🟡   BEKLE      — Karışık, acele etme')
print('  Skor ≥ 3.0 → 🔴   SAT        — Çoğunluk satış, kaçın')
print('  Skor <  3.0 → 🔴🔴 GÜÇLÜ SAT  — 4 modelin büyük çoğunluğu satış işaret ediyor')
print()
print('SÜTUN AÇIKLAMALARI:')
print('  TA      : Kural tabanlı TA puanı (0-9)')
print('  YZ_%    : Yapay Zekanın (XGBoost) yükseliş olasılığı tahmini')
print('  GARCH   : Oynaklık rejimi — DüşükVol=fırsat, YüksekVol=kaçın')
print('  Patern  : Geçmişteki benzer desenler sonrası beklenen 3-bar getiri')
print('  Stop    : ATR×1.5 tabanlı önerilen zarar kes seviyesi')
print('  Hedef   : ATR×2.5 tabanlı önerilen kar al seviyesi')

# Sinyal dağılımı
print('\n' + '─' * W2)
print('SİNYAL DAĞILIMI (tüm tarama):')
for sinyal, cnt in full_df['Sinyal'].value_counts().items():
    bar = '█' * max(1, cnt // 6)
    print(f'  {sinyal:<20}: {cnt:>4} hisse ({cnt/len(full_df)*100:5.1f}%)  {bar}')

# Hacim + YZ kombinasyonu (en yüksek güven)
combo = full_df[(full_df['HacimOran'] > 1.3) & (full_df['YZ_%'] >= 60)].nlargest(10, 'Skor')
if len(combo) > 0:
    print('\n' + '─' * W2)
    print('YÜKSEk HACİM + YÜKSEK YZ GUVEN — En güçlü kombinasyon:')
    print(combo[['Hisse','Fiyat','Skor','Sinyal','YZ_%','HacimOran','GARCH','Patern']].to_string(index=False))
    print('(Bu hisseler hem YZ hem piyasa hacmiyle desteklenen fırsatları gösterir)')


# Tahminleri kaydet (bir sonraki çalıştırmada doğrulanacak)
log_scan_predictions(full_df, INTERVAL_STR)


## 🔎 10. BENZERLİK PATERNİ TARAYICISI (DTW Tabanlı)

In [ ]:
from scipy.spatial.distance import euclidean

def find_similar_patterns(df: pd.DataFrame, lookback: int = 20,
                           n_matches: int = 5) -> pd.DataFrame:
    """
    Son 'lookback' barı referans al, geçmişte en benzer desenleri bul.
    Normalize edilmiş Öklid mesafesi kullanır (hızlı DTW yerine).
    Eşleşme sonrası getiriyi döndürür → istatistiksel ön görü.
    """
    prices = df['Close'].values
    rets   = df['Log_Return'].dropna().values

    # Referans desen: son lookback barın normalize getirileri
    ref_window = rets[-lookback:]
    ref_norm   = (ref_window - ref_window.mean()) / (ref_window.std() + 1e-9)

    matches = []
    horizon = 5   # ileriye kaç bar bak

    for i in range(lookback, len(rets) - lookback - horizon):
        window = rets[i-lookback:i]
        norm   = (window - window.mean()) / (window.std() + 1e-9)
        dist   = np.linalg.norm(ref_norm - norm)   # L2
        future_ret = np.sum(rets[i:i+horizon])      # sonraki horizon bar getirisi
        matches.append({'idx': i, 'dist': dist,
                        'date': df.index[i].date(),
                        f'Sonraki {horizon} bar': round(future_ret * 100, 2)})

    res = pd.DataFrame(matches).sort_values('dist').head(n_matches)
    res = res[['date', 'dist', f'Sonraki {horizon} bar']]
    res.columns = ['Tarih', 'Mesafe', f'Sonraki {horizon}bar Getiri %']

    avg_fwd = res[f'Sonraki {horizon}bar Getiri %'].mean()
    direction = 'YUKARI ▲' if avg_fwd > 0 else 'AŞAĞI ▼'

    print(f'\n[PATERİN ANALİZİ] Son {lookback} bar ile en benzer {n_matches} geçmiş desen:')
    print(res.to_string(index=False))
    print(f'\n→ Ortalama sonraki {horizon} bar getiri tahmini: {avg_fwd:+.2f}% ({direction})')

    return res


pattern_res = find_similar_patterns(df.dropna(), lookback=20, n_matches=5)

## 📋 12. HERKES İÇİN ANLAŞILIR KARAR RAPORU — "Ne Yapmalıyım?"

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  ANLAŞILIR KARAR PANELİ
#  Teknik bilgisi olmayan birine de ne yapması gerektiğini söyler.
# ══════════════════════════════════════════════════════════════════

def interpret_rsi(rsi):
    if rsi > 70:
        return ('🔴 AŞIRI ALINDI', 'Fiyat çok hızlı yükseldi, düzeltme gelebilir. Yeni alım için bekle.')
    elif rsi > 55:
        return ('🟡 GÜÇLÜ BÖLGE',  'Fiyat yükseliş momentumunda ama dikkatli ol.')
    elif rsi > 45:
        return ('🟢 NÖTR',          'Ne çok yüksek ne çok düşük. Trend yönü belirleyici.')
    elif rsi > 30:
        return ('🟡 ZAYIF BÖLGE',   'Fiyat düşüş baskısı altında.')
    else:
        return ('🔴 AŞIRI SATILDI', 'Fiyat çok hızlı düştü, toparlanma gelebilir. Dikkatli al.')

def interpret_macd(hist):
    if hist > 0.01:
        return ('🟢 AL YÖNÜNDE',  'Kısa vadeli yükseliş ivmesi var.')
    elif hist > 0:
        return ('🟡 ZAYIF AL',    'Çok küçük bir yükseliş işareti, doğrulama bekle.')
    elif hist > -0.01:
        return ('🟡 ZAYIF SAT',   'Çok küçük bir düşüş işareti.')
    else:
        return ('🔴 SAT YÖNÜNDE', 'Kısa vadeli düşüş ivmesi var.')

def interpret_adx(adx):
    if adx > 40:
        return ('💪 ÇOK GÜÇLÜ TREND', 'Piyasa güçlü bir yönde ilerliyor. Trend takipçi stratejiler işe yarar.')
    elif adx > 25:
        return ('✅ TREND VAR',        'Belirgin bir yön var, sinyal güvenilirliği yüksek.')
    elif adx > 20:
        return ('⚠️ ZAYIF TREND',      'Trend oluşuyor, henüz net değil.')
    else:
        return ('❌ YATAY PİYASA',     'Belirgin yön yok. Scalp sinyalleri daha az güvenilir.')

def interpret_vol_ratio(vr):
    if vr > 2.0:
        return ('🔥 PATLAMACI HACİM', 'Normal günün 2 katı hacim var! Büyük bir hareket yaklaşıyor olabilir.')
    elif vr > 1.3:
        return ('📈 YÜKSEK HACİM',    'Ortalamanın üzerinde hacim. Hareket destekli.')
    elif vr > 0.7:
        return ('➡️  NORMAL HACİM',   'Sıradan bir gün. Sinyaller orta güvenilirlikte.')
    else:
        return ('📉 DÜŞÜK HACİM',     'Hacim yok, piyasa uyuşuk. Sahte hareketlere dikkat.')

def interpret_strategy_score(winrate, ret):
    if winrate >= 60 and ret > 5:
        return '🏆 MÜKEMMEL — Bu strateji hem çok kazandırıyor hem doğru tahmin ediyor.'
    elif winrate >= 55 and ret > 0:
        return '✅ İYİ — Karlı ve güvenilir. Canlı kullanıma uygundur.'
    elif winrate >= 50 and ret > 0:
        return '⚠️ ORTA — Karlı ama tahminlerde bazen yanılıyor. Dikkatli kullan.'
    elif ret > 0:
        return '🤔 ZAYIF TAHMIN — Para kazandı ama şansa bağlı olabilir.'
    else:
        return '❌ KÖTÜ — Bu periyotta para kaybettirdi. Kullanma.'

def get_overall_signal(sig_a_str, sig_c_str, adx_val):
    a_bull = 'AL' in sig_a_str
    c_bull = sig_c_str and 'AL' in sig_c_str
    a_bear = 'SAT' in sig_a_str
    c_bear = sig_c_str and 'SAT' in sig_c_str
    trend_ok = adx_val > 20

    if a_bull and c_bull and trend_ok:
        return ('🟢🟢🟢 GÜÇLÜ AL', 'Her iki sistem de ALIŞ diyor ve piyasada belirgin bir yükseliş trendi var. '
                                   'Bu, sinyal kalitesinin en yüksek olduğu durumdur.')
    elif a_bull and c_bull:
        return ('🟢🟢 AL (Trend Zayıf)', 'Her iki sistem alış diyor ancak trend gücü düşük. '
                                          'Daha küçük pozisyonla gir.')
    elif (a_bull or c_bull) and not (a_bear or c_bear):
        return ('🟡 ZAYIF AL', 'Sistemlerden sadece biri alış diyor. '
                                'Ek onay beklemeden işlem açma.')
    elif a_bear and c_bear and trend_ok:
        return ('🔴🔴🔴 GÜÇLÜ SAT/BEKLE', 'Her iki sistem de SATIŞ diyor ve belirgin bir düşüş trendi var. '
                                           'Pozisyon alma, mevcut pozisyonu kapat.')
    elif a_bear or c_bear:
        return ('🔴 SAT/BEKLE', 'En az bir sistem satiş diyor. '
                                 'Yeni alım yapma, temkinli ol.')
    else:
        return ('⚪ BEKLE', 'Sistemler net bir sinyal üretmiyor. '
                             'Fırsatı kaçırmak yerine beklemek daha akıllıcadır.')


# ── Mevcut bar değerlerini al ───────────────────────────────────
last_bar   = df.dropna().iloc[-1]
rsi_lbl,   rsi_desc   = interpret_rsi(last_bar['RSI'])
macd_lbl,  macd_desc  = interpret_macd(last_bar['MACD_Hist'])
adx_lbl,   adx_desc   = interpret_adx(last_bar['ADX'])
vol_lbl,   vol_desc   = interpret_vol_ratio(last_bar['Vol_Ratio'])

# A sinyali (son bar)
sig_a_last = sig_A.loc[last_bar.name] if last_bar.name in sig_A.index else 0
sig_a_txt  = 'AL ▲' if sig_a_last == 1 else ('SAT ▼' if sig_a_last == -1 else 'BEKLE ─')

# XGBoost son tahmin
avail_c = [c for c in FEATURE_COLS if c in df.columns]
last_feat = df[avail_c].dropna().iloc[-1:]
if len(last_feat) > 0:
    lf_sc      = xgb_scaler_opt.transform(last_feat)
    xgb_pred   = xgb_model_opt.predict(lf_sc)[0]
    xgb_proba  = xgb_model_opt.predict_proba(lf_sc)[0]
    sig_c_txt  = f'AL ▲ (%{xgb_proba[1]*100:.0f} güven)' if xgb_pred == 1 else f'SAT ▼ (%{xgb_proba[0]*100:.0f} güven)'
else:
    sig_c_txt = 'Veri yok'

overall_lbl, overall_desc = get_overall_signal(sig_a_txt, sig_c_txt, last_bar['ADX'])

# Scalp parametreleri
close_px = last_bar['Close']
atr_px   = last_bar['ATR']
stop_px  = round(close_px - 1.5 * atr_px, 2)
tgt_px   = round(close_px + 2.5 * atr_px, 2)
rr_ratio = round((tgt_px - close_px) / max(close_px - stop_px, 0.01), 2)

# ── EKRANA YAZ ─────────────────────────────────────────────────
W = 68
print('╔' + '═'*W + '╗')
print('║' + f'  {TICKER} — GÜNCEL PİYASA DURUMU VE KARAR RAPORU'.center(W) + '║')
print('║' + f'  {datetime.now().strftime("%d %B %Y, %H:%M")} | Zaman Dilimi: {INTERVAL_STR}'.center(W) + '║')
print('╠' + '═'*W + '╣')

print('║' + '  📊 GÖSTERGE ANALİZİ'.ljust(W) + '║')
print('╠' + '─'*W + '╣')
print(f'║  RSI (Aşırı Alım/Satım Göstergesi)   : {rsi_lbl:<30}║')
print(f'║    → {rsi_desc:<{W-5}}║')
print(f'║  MACD (Momentum Göstergesi)           : {macd_lbl:<30}║')
print(f'║    → {macd_desc:<{W-5}}║')
print(f'║  ADX (Trend Gücü)                     : {adx_lbl:<30}║')
print(f'║    → {adx_desc:<{W-5}}║')
print(f'║  Hacim (İşlem Yoğunluğu)              : {vol_lbl:<30}║')
print(f'║    → {vol_desc:<{W-5}}║')

print('╠' + '═'*W + '╣')
print('║' + '  🤖 STRATEJİ KARARLARI'.ljust(W) + '║')
print('╠' + '─'*W + '╣')
print(f'║  Klasik TA Sistemi (Kural Tabanlı)    : {sig_a_txt:<30}║')
print(f'║  Yapay Zeka (XGBoost, Optimize)       : {sig_c_txt:<30}║')
print('╠' + '═'*W + '╣')
print('║' + f'  GENEL KARAR: {overall_lbl}'.ljust(W) + '║')
print(f'║  {overall_desc[:W-3]:<{W-2}}║')
if len(overall_desc) > W-3:
    print(f'║  {overall_desc[W-3:2*(W-3)]:<{W-2}}║')
print('╠' + '═'*W + '╣')
print('║' + '  💰 SCALP İŞLEM ÖNERİSİ (ATR Tabanlı)'.ljust(W) + '║')
print('╠' + '─'*W + '╣')
print(f'║  Şu anki fiyat   : {close_px:.2f} TL{"":<{W-26}}║')
print(f'║  Stop Loss       : {stop_px:.2f} TL  ← Buraya gelirse zararı kes!{"":<{W-55}}║')
print(f'║  Hedef Fiyat     : {tgt_px:.2f} TL  ← Burada kar al{"":<{W-47}}║')
print(f'║  Risk/Ödül Oranı : 1:{rr_ratio}  ← 1 birim riske {rr_ratio} birim kazanç{"":<{W-55}}║')
print('╠' + '═'*W + '╣')
print('║' + '  📈 BACKTEST ÖZETI (Geçmiş Test Sonuçları)'.ljust(W) + '║')
print('╠' + '─'*W + '╣')

all_res_display = [res_A, res_B, res_C, res_C_opt]
for r in all_res_display:
    wr   = r['Win Rate %']
    ret  = r['Toplam Getiri %']
    yorum = interpret_strategy_score(wr, ret)
    name  = r['Strateji'][:25]
    print(f'║  {name:<26}: Getiri {ret:+6.1f}% | Win Rate {wr:5.1f}% ║')
    print(f'║    → {yorum[:W-6]:<{W-6}}║')

print('╠' + '═'*W + '╣')
print('║' + f'  🏆 ŞAMPİYON: {champ_full["Strateji"]}'.ljust(W) + '║')

# Öğrenme istatistiği
wf_avg = wf_results['Doğruluk %'].mean()
print('╠' + '═'*W + '╣')
print('║' + '  🧠 MODELİN ÖĞRENMESİ'.ljust(W) + '║')
print('╠' + '─'*W + '╣')
print(f'║  Walk-Forward Ortalama Doğruluk : %{wf_avg:.1f}{"":<{W-38}}║')
wf_interp = ('✅ Model güvenilir tahmin yapıyor.' if wf_avg >= 55
             else '⚠️ Tahminler sınırda, daha fazla veriye ihtiyaç var.')
print(f'║  Yorum: {wf_interp:<{W-9}}║')
print(f'║  Toplam geçmiş çalıştırma       : {len(history)}{"":<{W-37}}║')
print(f'║  Model trend durumu             : {performance_trend(history)[:30]:<30}║')
print('╚' + '═'*W + '╝')


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  GÖRSEL DASHBOARD — Tek bakışta her şeyi anlayan grafik
# ══════════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(18, 14), facecolor='#0d1117')
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel 1: Gösterge Trafik Işıkları ─────────────────────────
ax_lights = fig.add_subplot(gs[0, 0])
ax_lights.set_facecolor('#0d1117')
ax_lights.set_xlim(0, 4); ax_lights.set_ylim(0, 5)
ax_lights.axis('off')
ax_lights.set_title('📊 Göstergeler', color='white', fontsize=12, fontweight='bold', pad=10)

indicators = [
    ('RSI', last_bar['RSI'], rsi_lbl),
    ('MACD', last_bar['MACD_Hist'], macd_lbl),
    ('ADX', last_bar['ADX'], adx_lbl),
    ('Hacim', last_bar['Vol_Ratio'], vol_lbl),
]
color_map = {'🟢': '#00ff88', '🔴': '#ff4444', '🟡': '#ffd93d',
             '✅': '#00ff88', '❌': '#ff4444', '⚠': '#ffd93d',
             '💪': '#00ff88', '📈': '#00ff88', '📉': '#ff4444',
             '➡': '#aaaaaa', '🔥': '#ff9500'}

for i, (name, val, label) in enumerate(indicators):
    first_emoji = label[0]
    col = next((v for k, v in color_map.items() if k in first_emoji), '#aaaaaa')
    y = 4 - i
    circle = plt.Circle((0.4, y), 0.28, color=col, zorder=3)
    ax_lights.add_patch(circle)
    ax_lights.text(0.85, y,    name,  color='white', fontsize=10, va='center', fontweight='bold')
    ax_lights.text(0.85, y-0.3, f'{val:.1f}', color='#aaaaaa', fontsize=8, va='center')

# ── Panel 2: Genel Karar Kartı ─────────────────────────────────
ax_decision = fig.add_subplot(gs[0, 1:])
ax_decision.set_facecolor('#0d1117')
ax_decision.axis('off')
ax_decision.set_title('🎯 Karar & Scalp Parametreleri', color='white', fontsize=12, fontweight='bold', pad=10)

decision_color = '#00ff88' if 'AL' in overall_lbl else ('#ff4444' if 'SAT' in overall_lbl else '#ffd93d')
fancy_box = mpatches.FancyBboxPatch((0.02, 0.55), 0.96, 0.38,
    boxstyle='round,pad=0.02', facecolor=decision_color+'33',
    edgecolor=decision_color, linewidth=2, transform=ax_decision.transAxes)
ax_decision.add_patch(fancy_box)
ax_decision.text(0.5, 0.76, overall_lbl, color=decision_color,
                 ha='center', va='center', fontsize=16, fontweight='bold',
                 transform=ax_decision.transAxes)
ax_decision.text(0.5, 0.62, overall_desc[:80], color='white',
                 ha='center', va='center', fontsize=8, style='italic',
                 transform=ax_decision.transAxes, wrap=True)

# Scalp parametreleri
params_txt = (
    f'Giriş: {close_px:.2f} TL    '
    f'Stop: {stop_px:.2f} TL (-{(close_px-stop_px)/close_px:.1%})    '
    f'Hedef: {tgt_px:.2f} TL (+{(tgt_px-close_px)/close_px:.1%})    '
    f'R/Ö: 1:{rr_ratio}'
)
ax_decision.text(0.5, 0.35, params_txt, color='#ffd93d',
                 ha='center', va='center', fontsize=10, fontweight='bold',
                 transform=ax_decision.transAxes,
                 bbox=dict(boxstyle='round', facecolor='#1a1a2e', edgecolor='#ffd93d'))

ax_decision.text(0.5, 0.12,
    'Stop Loss: Fiyat buraya gelirse zararı sınırla | Hedef: Kar al ve çık',
    color='#888888', ha='center', va='center', fontsize=8, transform=ax_decision.transAxes)

# ── Panel 3: Kümülatif Getiri (tüm stratejiler) ───────────────
ax_ret = fig.add_subplot(gs[1, :2])
ax_ret.set_facecolor('#161b22')
ax_ret.set_title('📈 Kümülatif Getiri Karşılaştırması (Test Dönemi)', color='white', fontsize=11)
for i, r in enumerate([res_A, res_B, res_C, res_C_opt]):
    cum = r['_data']['Cum_Return']
    ax_ret.plot(cum.index, (cum.values - 1)*100,
                label=f"{r['Strateji']} ({r['Toplam Getiri %']:+.1f}%)",
                color=COLORS[i], linewidth=1.8)
bh = res_A['_data']['BH_Return']
ax_ret.plot(bh.index, (bh.values-1)*100, '--', color='gray', alpha=0.5, label='Al & Tut')
ax_ret.axhline(0, color='white', linewidth=0.5, alpha=0.3)
ax_ret.set_ylabel('Getiri (%)', color='white')
ax_ret.legend(fontsize=8, loc='upper left')
ax_ret.grid(alpha=0.12); ax_ret.tick_params(colors='white')
for sp in ax_ret.spines.values(): sp.set_color('#333')
for lb in ax_ret.get_xticklabels()+ax_ret.get_yticklabels(): lb.set_color('white')

# ── Panel 4: Walk-Forward Doğruluk Çubuğu ─────────────────────
ax_wf = fig.add_subplot(gs[1, 2])
ax_wf.set_facecolor('#161b22')
ax_wf.set_title('🧠 Walk-Forward\nDoğruluk %', color='white', fontsize=10)
wf_colors = ['#00ff88' if v >= 55 else ('#ffd93d' if v >= 50 else '#ff4444')
             for v in wf_results['Doğruluk %']]
bars = ax_wf.bar(wf_results['Tur'], wf_results['Doğruluk %'], color=wf_colors, alpha=0.85)
ax_wf.axhline(55, color='#00ff88', linewidth=1, linestyle='--', alpha=0.7, label='İyi (55%)')
ax_wf.axhline(50, color='#ffd93d', linewidth=1, linestyle='--', alpha=0.7, label='Ort (50%)')
ax_wf.set_ylim(40, 75); ax_wf.set_ylabel('%', color='white')
ax_wf.legend(fontsize=7); ax_wf.grid(alpha=0.12, axis='y')
ax_wf.tick_params(colors='white'); ax_wf.tick_params(axis='x', labelrotation=30, labelsize=8)
for sp in ax_wf.spines.values(): sp.set_color('#333')
for lb in ax_wf.get_xticklabels()+ax_wf.get_yticklabels(): lb.set_color('white')
for bar, val in zip(bars, wf_results['Doğruluk %']):
    ax_wf.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
               f'{val:.0f}%', ha='center', va='bottom', color='white', fontsize=8)

# ── Panel 5: Strateji Karşılaştırma (Win Rate vs Getiri) ──────
ax_scatter = fig.add_subplot(gs[2, :2])
ax_scatter.set_facecolor('#161b22')
ax_scatter.set_title('🎯 Win Rate vs Getiri — İdeal: sağ üst köşe', color='white', fontsize=11)
for i, r in enumerate([res_A, res_B, res_C, res_C_opt]):
    ax_scatter.scatter(r['Win Rate %'], r['Toplam Getiri %'],
                       s=220, color=COLORS[i], zorder=5, alpha=0.9,
                       edgecolors='white', linewidth=1.2)
    ax_scatter.annotate(r['Strateji'].split(':')[0],
                        (r['Win Rate %'], r['Toplam Getiri %']),
                        textcoords='offset points', xytext=(8, 4),
                        color=COLORS[i], fontsize=8)
ax_scatter.axvline(55, color='#00ff88', linestyle='--', alpha=0.4, linewidth=1)
ax_scatter.axhline(0,  color='white',   linestyle='--', alpha=0.3, linewidth=1)
ax_scatter.fill_betweenx(
    [max(0, min(r['Toplam Getiri %'] for r in [res_A,res_B,res_C,res_C_opt])-2),
     max(r['Toplam Getiri %'] for r in [res_A,res_B,res_C,res_C_opt])+2],
    55, 100, alpha=0.05, color='#00ff88')
ax_scatter.text(56, ax_scatter.get_ylim()[1]*0.9, 'İDEAL\nBÖLGE', color='#00ff88',
                fontsize=8, alpha=0.6)
ax_scatter.set_xlabel('Win Rate % (Doğruluk)', color='white')
ax_scatter.set_ylabel('Toplam Getiri %', color='white')
ax_scatter.grid(alpha=0.12); ax_scatter.tick_params(colors='white')
for sp in ax_scatter.spines.values(): sp.set_color('#333')
for lb in ax_scatter.get_xticklabels()+ax_scatter.get_yticklabels(): lb.set_color('white')

# ── Panel 6: Geçmiş Öğrenme Eğrisi ───────────────────────────
ax_hist = fig.add_subplot(gs[2, 2])
ax_hist.set_facecolor('#161b22')
ax_hist.set_title('📚 Modelin\nÖğrenme Geçmişi', color='white', fontsize=10)
if len(history) >= 2:
    runs   = [f"#{i+1}" for i in range(len(history))]
    wr_hist= [h.get('xgb_opt_winrate', h.get('xgb_winrate', 50)) for h in history]
    ax_hist.plot(runs, wr_hist, color='#c77dff', marker='o', linewidth=2, markersize=8)
    ax_hist.axhline(55, color='#00ff88', linestyle='--', alpha=0.5, linewidth=1)
    ax_hist.set_ylim(40, 75)
    ax_hist.fill_between(runs, wr_hist, 50, alpha=0.2,
                         color='#00ff88' if wr_hist[-1] >= 50 else '#ff4444')
    for i, (x, y) in enumerate(zip(runs, wr_hist)):
        ax_hist.annotate(f'{y:.0f}%', (x, y), textcoords='offset points',
                         xytext=(0, 6), ha='center', color='white', fontsize=8)
else:
    ax_hist.text(0.5, 0.5, 'İlk çalıştırma.\nGecmiş birikmeli\ngelişim görünür.',
                 ha='center', va='center', color='#aaaaaa', fontsize=10,
                 transform=ax_hist.transAxes)
ax_hist.set_ylabel('Win Rate %', color='white')
ax_hist.grid(alpha=0.12, axis='y'); ax_hist.tick_params(colors='white')
for sp in ax_hist.spines.values(): sp.set_color('#333')
for lb in ax_hist.get_xticklabels()+ax_hist.get_yticklabels(): lb.set_color('white')

fig.suptitle(
    f'{TICKER} — Kapsamlı Analiz Panosu | {datetime.now().strftime("%d %b %Y %H:%M")}',
    color='white', fontsize=14, fontweight='bold', y=0.99
)
plt.savefig('dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Dashboard kaydedildi: dashboard.png')
